# Banking Assignment — Exception Handling (100 Questions with Hints)

An unsolved assessment notebook using accounts, deposits, withdrawals, transfers, cards, loans, KYC, fraud monitoring, reconciliation, and audit trails.

**Structure:** 30 easy + 40 medium + 30 difficult = **100 questions**.


## Student instructions

Solve all questions using banking scenarios. Practice try/except, custom exceptions, chaining, cleanup, resilient batch processing. Do not store real customer data, credentials, PINs, or complete card numbers. Each question includes a graduated hint but intentionally has no solution.

**Recommended workflow:** write assumptions, implement the smallest correct function/class, test normal and edge cases, then explain your design.


---

## Easy

Fundamentals and direct applications.


### Q1. Parse a deposit amount for banking case 1; use sample reference `ACC-1001` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
from decimal import Decimal, InvalidOperation

def parse_deposit_amount(v):
    try:
        amt = Decimal(str(v))  # risky line -> InvalidOperation
    except (InvalidOperation, ValueError, TypeError) as e:
        raise ValueError("Invalid amount") from e # preserve cause
    if amt <= 0 or not amt.is_finite():
        raise ValueError(f"Business Error: amount must be >0, got {amt}")
    return amt # return only after validation -> protects balance

# Test for ACC-1001
print(parse_deposit_amount("1500.00")) # ACC-1001 SUCCESS

1500.00


### Q2. Handle a missing account key for banking case 1; use sample reference `ACC-1002` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
def get_balance(accounts, key):
    try:
        bal = accounts[key] # risky -> KeyError
    except KeyError as e:
        raise KeyError(f"[{key}] Business Error: Account not found") from e
    except TypeError:
        print("Programming Error")
        return
    return bal

# Test
db = {"ACC-1001": 5000, "ACC-1003": 3000}

# ACC-1002 is missing - will fail (expected)
try:
    print(get_balance(db, "ACC-1002"))
except Exception as e:
    print(f"ACC-1002 -> {e}")

# ACC-1001 exists - success
print(f"ACC-1001 -> {get_balance(db, 'ACC-1001')}")

ACC-1002 -> '[ACC-1002] Business Error: Account not found'
ACC-1001 -> 5000


### Q3. Prevent division by zero in average balance for banking case 1; use sample reference `ACC-1003` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

def avg_balance(total, months):
    try:
        avg = total / months  # risky -> ZeroDivisionError
    except ZeroDivisionError as e:
        raise ZeroDivisionError(f"[ACC-1003] Business Error: months is 0") from e
    except TypeError:
        print("Programming Error: total/months must be numbers")
        return
    return avg

# Test ACC-1003
print(avg_balance(12000, 12)) # success -> 1000
try:
    print(avg_balance(12000, 0)) # ACC-1003 failure case
except Exception as e:
    print(e)

1000.0
[ACC-1003] Business Error: months is 0


### Q4. Open a transaction file safely for banking case 1; use sample reference `ACC-1004` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

def withdraw(balance, amount):
    if amount < 0:
        raise ValueError(f"[ACC-1005] Business Error: Negative withdrawal {amount} not allowed")
    if amount > balance:
        raise ValueError(f"[ACC-1005] Business Error: Insufficient balance")
    return balance - amount # only after check -> protects balance

# Test
print(withdraw(5000, 1000)) # success -> 4000
try:
    withdraw(5000, -500) # ACC-1005 failure
except Exception as e:
    print(e)

4000
[ACC-1005] Business Error: Negative withdrawal -500 not allowed


### Q5. Reject a negative withdrawal for banking case 1; use sample reference `ACC-1005` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
def withdraw(balance, amount):
    if amount < 0:
        raise ValueError(f"[ACC-1005] Business Error: Negative withdrawal {amount} not allowed")
    if amount > balance:
        raise ValueError(f"[ACC-1005] Business Error: Insufficient balance")
    return balance - amount # only after check -> protects balance

# Test
print(withdraw(5000, 1000)) # success -> 4000
try:
    withdraw(5000, -500) # ACC-1005 failure
except Exception as e:
    print(e)

4000
[ACC-1005] Business Error: Negative withdrawal -500 not allowed


### Q6. Catch invalid tenure input for banking case 1; use sample reference `ACC-1006` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

def check_tenure(tenure):
    try:
        t = int(tenure) # risky -> ValueError
    except (ValueError, TypeError) as e:
        raise ValueError(f"[ACC-1006] Business Error: Invalid tenure {tenure!r}") from e

    if t <= 0:
        raise ValueError(f"[ACC-1006] Business Error: Tenure must be >0")
    return t

# Test
print(check_tenure("5")) # success
try:
    print(check_tenure("abc")) # ACC-1006 failure
except Exception as e:
    print(e)

5
[ACC-1006] Business Error: Invalid tenure 'abc'


### Q7. Use else after pin-format validation for banking case 1; use sample reference `ACC-1007` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

def check_pin(pin):
    try:
        p = str(pin) # risky -> ValueError if not string convertible
        if not p.isdigit():
            raise ValueError("Pin must be digits")
    except (ValueError, TypeError) as e:
        raise ValueError(f"[ACC-1007] Business Error: Invalid pin {pin!r}") from e
    else:
        # else runs ONLY if no exception - this is pin-format validation pass
        if len(p) != 4:
            raise ValueError(f"[ACC-1007] Business Error: Pin must be 4 digits, got {len(p)}")
        return f"[ACC-1007] SUCCESS: Pin {p} validated"
    finally:
        # protects partially updated balances - always cleanup
        pass

# Test
print(check_pin("1234")) # ACC-1007 success
try:
    print(check_pin("12ab")) # fail
except Exception as e:
    print(e)

try:
    print(check_pin("12")) # boundary fail
except Exception as e:
    print(e)

[ACC-1007] SUCCESS: Pin 1234 validated
[ACC-1007] Business Error: Invalid pin '12ab'
[ACC-1007] Business Error: Pin must be 4 digits, got 2


### Q8. Use finally to close a session for banking case 1; use sample reference `ACC-1008` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

def close_session_example(acc_id):
    session = None
    try:
        session = {"acc": acc_id, "open": True}
        print(f"[{acc_id}] Session opened")
        if acc_id == "ACC-1008":
            # risky statement
            x = 10 / 0 # simulate failure -> ZeroDivisionError
    except ZeroDivisionError as e:
        print(f"[{acc_id}] Business Error: Transaction failed")
        raise ZeroDivisionError(f"[{acc_id}] Failed") from e
    finally:
        # finally ALWAYS runs -> protects partially updated balance + closes session
        if session:
            session["open"] = False
            print(f"[{acc_id}] Session closed in finally - Balance protected")

# Test ACC-1008
try:
    close_session_example("ACC-1008")
except Exception as e:
    print(f"Result: {e} | cause preserved: {e.__cause__}")

print("\nSuccess case:")
def success_case(acc_id):
    session = {"acc": acc_id, "open": True}
    try:
        print(f"[{acc_id}] Session opened")
        print(f"[{acc_id}] SUCCESS: Work done")
    finally:
        session["open"] = False
        print(f"[{acc_id}] Session closed in finally")

success_case("ACC-1008")

[ACC-1008] Session opened
[ACC-1008] Business Error: Transaction failed
[ACC-1008] Session closed in finally - Balance protected
Result: [ACC-1008] Failed | cause preserved: division by zero

Success case:
[ACC-1008] Session opened
[ACC-1008] SUCCESS: Work done
[ACC-1008] Session closed in finally


### Q9. Catch multiple numeric conversion errors for banking case 1; use sample reference `ACC-1009` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
def parse_amount(value):
    try:
        amount = float(value)  # smallest risky statement -> ValueError, TypeError
    except (ValueError, TypeError) as e:
        raise ValueError(f"[ACC-1009] Business Error: Invalid amount {value!r}") from e

    if amount <= 0:
        raise ValueError(f"[ACC-1009] Business Error: Amount must be >0")

    return amount

# Tests - empty, invalid type, boundary, realistic failure
for x in ["100", "", "abc", None, "0", "-5"]:
    try:
        print(f"{x!r} -> {parse_amount(x)}")
    except Exception as e:
        print(f"{x!r} -> {e} | cause preserved: {e.__cause__}")

'100' -> 100.0
'' -> [ACC-1009] Business Error: Invalid amount '' | cause preserved: could not convert string to float: ''
'abc' -> [ACC-1009] Business Error: Invalid amount 'abc' | cause preserved: could not convert string to float: 'abc'
None -> [ACC-1009] Business Error: Invalid amount None | cause preserved: float() argument must be a string or a real number, not 'NoneType'
'0' -> [ACC-1009] Business Error: Amount must be >0 | cause preserved: None
'-5' -> [ACC-1009] Business Error: Amount must be >0 | cause preserved: None


### Q10. Raise a clear valueerror for banking case 1; use sample reference `ACC-1010` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
def banking_case_1(value):
    try:
        amount = float(value)  # smallest risky -> ValueError, TypeError
    except (ValueError, TypeError) as e:
        raise ValueError(f"[ACC-1010] Business Error: Invalid banking input {value!r}") from e

    if amount <= 0:
        raise ValueError(f"[ACC-1010] Business Error: Banking case 1 - Amount must be >0")

    return amount  # protects partially updated balance - return only after validation

# Edge cases to test
for x in ["500", "", "abc", None, "0", "-100"]:
    try:
        print(f"{x!r} -> {banking_case_1(x)} SUCCESS")
    except Exception as e:
        print(f"{x!r} -> {e} | cause: {e.__cause__}")

'500' -> 500.0 SUCCESS
'' -> [ACC-1010] Business Error: Invalid banking input '' | cause: could not convert string to float: ''
'abc' -> [ACC-1010] Business Error: Invalid banking input 'abc' | cause: could not convert string to float: 'abc'
None -> [ACC-1010] Business Error: Invalid banking input None | cause: float() argument must be a string or a real number, not 'NoneType'
'0' -> [ACC-1010] Business Error: Banking case 1 - Amount must be >0 | cause: None
'-100' -> [ACC-1010] Business Error: Banking case 1 - Amount must be >0 | cause: None


### Q11. Parse a deposit amount for banking case 2; use sample reference `ACC-1011` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

from decimal import Decimal, InvalidOperation

class DepositError(Exception):
    pass

def parse_deposit_amount(value):
    try:
        return Decimal(str(value))
    except (InvalidOperation, ValueError, TypeError) as exc:
        raise DepositError("Invalid deposit amount") from exc

reference = "ACC-1011"

try:
    amount = parse_deposit_amount("1500.00")

    if amount <= 0:
        raise DepositError("Deposit must be greater than zero")

    print(f"{reference}: deposit amount = {amount}")

except DepositError as exc:
    print(f"{reference}: {exc}")

ACC-1011: deposit amount = 1500.00


### Q12. Handle a missing account key for banking case 2; use sample reference `ACC-1012` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class AccountNotFoundError(Exception):
    pass


def get_account(accounts, reference):
    try:
        return accounts[reference]
    except KeyError as exc:
        raise AccountNotFoundError(
            f"Account {reference} not found"
        ) from exc


accounts = {
    "ACC-1011": {"balance": 5000.00}
}

reference = "ACC-1012"

try:
    account = get_account(accounts, reference)
    print(f"{reference}: account found")
except AccountNotFoundError as exc:
    print(f"{reference}: {exc}")

ACC-1012: Account ACC-1012 not found


### Q13. Prevent division by zero in average balance for banking case 2; use sample reference `ACC-1013` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class AverageBalanceError(Exception):
    pass


def average_balance(total_balance, number_of_days):
    try:
        return total_balance / number_of_days
    except ZeroDivisionError as exc:
        raise AverageBalanceError(
            "Cannot calculate average balance: number of days is zero"
        ) from exc


reference = "ACC-1013"

try:
    result = average_balance(5000, 0)
    print(f"{reference}: average balance = {result}")
except AverageBalanceError as exc:
    print(f"{reference}: {exc}")

ACC-1013: Cannot calculate average balance: number of days is zero


### Q14. Open a transaction file safely for banking case 2; use sample reference `ACC-1014` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class TransactionFileError(Exception):
    pass


def open_transaction_file(filename):
    try:
        return open(filename, "r")
    except (FileNotFoundError, PermissionError, OSError) as exc:
        raise TransactionFileError(
            f"Unable to open transaction file: {filename}"
        ) from exc


reference = "ACC-1014"
filename = "transactions.txt"

try:
    file = open_transaction_file(filename)
    print(f"{reference}: transaction file opened successfully")
    file.close()
except TransactionFileError as exc:
    print(f"{reference}: {exc}")

ACC-1014: Unable to open transaction file: transactions.txt


### Q15. Reject a negative withdrawal for banking case 2; use sample reference `ACC-1015` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class WithdrawalError(Exception):
    pass


def withdraw(balance, amount):
    if amount < 0:
        raise WithdrawalError("Withdrawal amount cannot be negative")

    if amount > balance:
        raise WithdrawalError("Insufficient balance")

    return balance - amount


reference = "ACC-1015"
balance = 5000.00
withdrawal = -100.00

try:
    new_balance = withdraw(balance, withdrawal)
    print(f"{reference}: new balance = {new_balance}")
except WithdrawalError as exc:
    print(f"{reference}: {exc}")

ACC-1015: Withdrawal amount cannot be negative


### Q16. Catch invalid tenure input for banking case 2; use sample reference `ACC-1016` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class TenureError(Exception):
    pass


def parse_tenure(value):
    try:
        tenure = int(value)
    except (ValueError, TypeError) as exc:
        raise TenureError("Invalid tenure input") from exc

    if tenure <= 0:
        raise TenureError("Tenure must be greater than zero")

    return tenure


reference = "ACC-1016"
tenure_input = "abc"

try:
    tenure = parse_tenure(tenure_input)
    print(f"{reference}: tenure = {tenure} years")
except TenureError as exc:
    print(f"{reference}: {exc}")

ACC-1016: Invalid tenure input


### Q17. Use else after pin-format validation for banking case 2; use sample reference `ACC-1017` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class PinError(Exception):
    pass


def validate_pin(pin):
    try:
        if len(pin) != 4 or not pin.isdigit():
            raise PinError("Invalid PIN format")
    except TypeError as exc:
        raise PinError("PIN must be a string") from exc
    else:
        return True


reference = "ACC-1017"
pin = "1234"

try:
    if validate_pin(pin):
        print(f"{reference}: PIN format is valid")
except PinError as exc:
    print(f"{reference}: {exc}")

ACC-1017: PIN format is valid


### Q18. Use finally to close a session for banking case 2; use sample reference `ACC-1018` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class SessionError(Exception):
    pass


def process_session(session):
    try:
        if session is None:
            raise SessionError("Invalid session")
        print("Session processed successfully")
    finally:
        if session is not None:
            session.close()


class Session:
    def close(self):
        print("Session closed")


reference = "ACC-1018"
session = Session()

try:
    process_session(session)
    print(f"{reference}: session completed")
except SessionError as exc:
    print(f"{reference}: {exc}")

Session processed successfully
Session closed
ACC-1018: session completed


### Q19. Catch multiple numeric conversion errors for banking case 2; use sample reference `ACC-1019` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

id="19375"
class NumericInputError(Exception):
    pass


def parse_amount(value):
    try:
        amount = float(value)
    except (ValueError, TypeError) as exc:
        raise NumericInputError("Invalid numeric amount") from exc

    if amount < 0:
        raise NumericInputError("Amount cannot be negative")

    return amount


reference = "ACC-1019"
amount_input = "abc"

try:
    amount = parse_amount(amount_input)
    print(f"{reference}: amount = {amount}")
except NumericInputError as exc:
    print(f"{reference}: {exc}")

ACC-1019: Invalid numeric amount


### Q20. Raise a clear valueerror for banking case 2; use sample reference `ACC-1020` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class ValueErrorExample(Exception):
    pass


def validate_amount(amount):
    if amount <= 0:
        raise ValueError("Deposit amount must be greater than zero")
    return amount


reference = "ACC-1020"
amount = -500

try:
    result = validate_amount(amount)
    print(f"{reference}: valid amount = {result}")
except ValueError as exc:
    print(f"{reference}: {exc}")

ACC-1020: Deposit amount must be greater than zero


### Q21. Parse a deposit amount for banking case 3; use sample reference `ACC-1021` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

from decimal import Decimal, InvalidOperation

class DepositError(Exception):
    pass

def parse_deposit_amount(value):
    try:
        amount = Decimal(str(value))
    except (InvalidOperation, ValueError, TypeError) as exc:
        raise DepositError("Invalid deposit amount") from exc

    if amount <= 0:
        raise DepositError("Deposit amount must be greater than zero")

    return amount


reference = "ACC-1021"
deposit_input = "1500.00"

try:
    amount = parse_deposit_amount(deposit_input)
    print(f"{reference}: deposit amount = {amount}")
except DepositError as exc:
    print(f"{reference}: {exc}")

ACC-1021: deposit amount = 1500.00


### Q22. Handle a missing account key for banking case 3; use sample reference `ACC-1022` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class AccountNotFoundError(Exception):
    pass


def get_account(accounts, reference):
    try:
        return accounts[reference]
    except KeyError as exc:
        raise AccountNotFoundError(
            f"Account {reference} not found"
        ) from exc


accounts = {
    "ACC-1021": {"balance": 5000.00}
}

reference = "ACC-1022"

try:
    account = get_account(accounts, reference)
    print(f"{reference}: account found - {account}")
except AccountNotFoundError as exc:
    print(f"{reference}: {exc}")

ACC-1022: Account ACC-1022 not found


### Q23. Prevent division by zero in average balance for banking case 3; use sample reference `ACC-1023` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class AverageBalanceError(Exception):
    pass


def average_balance(total_balance, number_of_days):
    try:
        return total_balance / number_of_days
    except ZeroDivisionError as exc:
        raise AverageBalanceError(
            "Cannot calculate average balance: number of days is zero"
        ) from exc


reference = "ACC-1023"

try:
    result = average_balance(5000, 0)
    print(f"{reference}: average balance = {result}")
except AverageBalanceError as exc:
    print(f"{reference}: {exc}")

ACC-1023: Cannot calculate average balance: number of days is zero


### Q24. Open a transaction file safely for banking case 3; use sample reference `ACC-1024` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class TransactionFileError(Exception):
    pass


def open_transaction_file(filename):
    try:
        return open(filename, "r")
    except (FileNotFoundError, PermissionError, OSError) as exc:
        raise TransactionFileError(
            f"Unable to open transaction file: {filename}"
        ) from exc


reference = "ACC-1024"
filename = "transactions.txt"

try:
    file = open_transaction_file(filename)
    print(f"{reference}: transaction file opened successfully")
    file.close()
except TransactionFileError as exc:
    print(f"{reference}: {exc}")

ACC-1024: Unable to open transaction file: transactions.txt


### Q25. Reject a negative withdrawal for banking case 3; use sample reference `ACC-1025` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
class WithdrawalError(Exception):
    pass


def withdraw(balance, amount):
    if amount < 0:
        raise WithdrawalError("Withdrawal amount cannot be negative")

    if amount > balance:
        raise WithdrawalError("Insufficient balance")

    return balance - amount


reference = "ACC-1025"
balance = 5000.00
withdrawal = -100.00

try:
    new_balance = withdraw(balance, withdrawal)
    print(f"{reference}: new balance = {new_balance}")
except WithdrawalError as exc:
    print(f"{reference}: {exc}")

ACC-1025: Withdrawal amount cannot be negative


### Q26. Catch invalid tenure input for banking case 3; use sample reference `ACC-1026` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class TenureError(Exception):
    pass


def parse_tenure(value):
    try:
        tenure = int(value)
    except (ValueError, TypeError) as exc:
        raise TenureError("Invalid tenure input") from exc

    if tenure <= 0:
        raise TenureError("Tenure must be greater than zero")

    return tenure


reference = "ACC-1026"
tenure_input = "abc"

try:
    tenure = parse_tenure(tenure_input)
    print(f"{reference}: tenure = {tenure} years")
except TenureError as exc:
    print(f"{reference}: {exc}")

ACC-1026: Invalid tenure input


### Q27. Use else after pin-format validation for banking case 3; use sample reference `ACC-1027` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class PinError(Exception):
    pass


def validate_pin(pin):
    try:
        if len(pin) != 4 or not pin.isdigit():
            raise PinError("Invalid PIN format")
    except TypeError as exc:
        raise PinError("PIN must be a string") from exc
    else:
        return True


reference = "ACC-1027"
pin = "1234"

try:
    if validate_pin(pin):
        print(f"{reference}: PIN format is valid")
except PinError as exc:
    print(f"{reference}: {exc}")

ACC-1027: PIN format is valid


### Q28. Use finally to close a session for banking case 3; use sample reference `ACC-1028` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class SessionError(Exception):
    pass


class Session:
    def close(self):
        print("Session closed")


def process_session(session):
    try:
        if session is None:
            raise SessionError("Invalid session")

        print("Session processed successfully")

    finally:
        if session is not None:
            session.close()


reference = "ACC-1028"
session = Session()

try:
    process_session(session)
    print(f"{reference}: session completed")
except SessionError as exc:
    print(f"{reference}: {exc}")

Session processed successfully
Session closed
ACC-1028: session completed


### Q29. Catch multiple numeric conversion errors for banking case 3; use sample reference `ACC-1029` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class NumericInputError(Exception):
    pass


def parse_amount(value):
    try:
        amount = float(value)
    except (ValueError, TypeError) as exc:
        raise NumericInputError("Invalid numeric amount") from exc

    if amount < 0:
        raise NumericInputError("Amount cannot be negative")

    return amount


reference = "ACC-1029"
amount_input = "abc"

try:
    amount = parse_amount(amount_input)
    print(f"{reference}: amount = {amount}")
except NumericInputError as exc:
    print(f"{reference}: {exc}")

ACC-1029: Invalid numeric amount


### Q30. Raise a clear valueerror for banking case 3; use sample reference `ACC-1030` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

def validate_amount(amount):
    if amount <= 0:
        raise ValueError("Deposit amount must be greater than zero")
    return amount


reference = "ACC-1030"
amount = -500

try:
    result = validate_amount(amount)
    print(f"{reference}: valid amount = {result}")
except ValueError as exc:
    print(f"{reference}: {exc}")

ACC-1030: Deposit amount must be greater than zero



---

## Medium

Reusable components and multi-step workflows.


### Q31. Create insufficientfundserror for banking case 1; use sample reference `ACC-1031` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class InsufficientFundsError(Exception):
    pass


def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError("Insufficient funds")

    return balance - amount


reference = "ACC-1031"
balance = 5000.00
withdrawal = 6000.00

try:
    new_balance = withdraw(balance, withdrawal)
    print(f"{reference}: new balance = {new_balance}")
except InsufficientFundsError as exc:
    print(f"{reference}: {exc}")

ACC-1031: Insufficient funds


### Q32. Chain invalidtransactionerror from valueerror for banking case 1; use sample reference `ACC-1032` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Write your solution here
class InvalidTransactionError(Exception):
    pass


def parse_transaction_amount(value):
    try:
        return float(value)
    except (ValueError, TypeError) as exc:
        raise InvalidTransactionError("Invalid transaction amount") from exc


reference = "ACC-1032"
amount_input = "abc"

try:
    amount = parse_transaction_amount(amount_input)
    print(f"{reference}: transaction amount = {amount}")
except InvalidTransactionError as exc:
    print(f"{reference}: {exc}")

ACC-1032: Invalid transaction amount


### Q33. Process transfers while isolating bad rows for banking case 1; use sample reference `ACC-1033` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class TransferError(Exception):
    pass


def process_transfer(row):
    try:
        from_account = row["from"]
        to_account = row["to"]
        amount = float(row["amount"])
    except (KeyError, ValueError, TypeError) as exc:
        raise TransferError("Invalid transfer row") from exc

    if amount <= 0:
        raise TransferError("Transfer amount must be greater than zero")

    return f"Transfer {amount} from {from_account} to {to_account}"


reference = "ACC-1033"

rows = [
    {"from": "ACC-1001", "to": "ACC-1002", "amount": "500"},
    {"from": "ACC-1003", "to": "ACC-1004", "amount": "abc"},
    {"from": "ACC-1005", "to": "ACC-1006", "amount": "200"}
]

for row in rows:
    try:
        result = process_transfer(row)
        print(f"{reference}: {result}")
    except TransferError as exc:
        print(f"{reference}: skipped bad row - {exc}")

ACC-1033: Transfer 500.0 from ACC-1001 to ACC-1002
ACC-1033: skipped bad row - Invalid transfer row
ACC-1033: Transfer 200.0 from ACC-1005 to ACC-1006


### Q34. Retry a transient payment gateway for banking case 1; use sample reference `ACC-1034` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class PaymentGatewayError(Exception):
    pass


def payment_gateway(attempt):
    if attempt < 3:
        raise ConnectionError("Temporary gateway failure")
    return "Payment successful"


def process_payment(reference, amount):
    try:
        for attempt in range(1, 4):
            try:
                result = payment_gateway(attempt)
                return result
            except ConnectionError:
                if attempt == 3:
                    raise
    except ConnectionError as exc:
        raise PaymentGatewayError("Payment gateway failed after retries") from exc


reference = "ACC-1034"
amount = 1000.00

try:
    result = process_payment(reference, amount)
    print(f"{reference}: {result}")
except PaymentGatewayError as exc:
    print(f"{reference}: {exc}")

ACC-1034: Payment successful


### Q35. Log failed standing instructions for banking case 1; use sample reference `ACC-1035` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
class StandingInstructionError(Exception):
    pass


def process_instruction(instruction):
    try:
        amount = float(instruction["amount"])
    except (KeyError, ValueError, TypeError) as exc:
        raise StandingInstructionError("Invalid standing instruction") from exc

    if amount <= 0:
        raise StandingInstructionError(
            "Standing instruction amount must be greater than zero"
        )

    return amount


reference = "ACC-1035"

instruction = {"amount": "abc"}

try:
    amount = process_instruction(instruction)
    print(f"{reference}: instruction processed - {amount}")
except StandingInstructionError as exc:
    print(f"{reference}: FAILED - {exc}")

ACC-1035: FAILED - Invalid standing instruction


### Q36. Return a structured failure result for banking case 1; use sample reference `ACC-1036` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
def debit_account(account, amount):
    if not account or amount is None:
        return {
            "reference": "ACC-1036",
            "status": "FAILED",
            "error_code": "INVALID_INPUT",
            "message": "Invalid input",
            "balance_updated": False
        }

    if not isinstance(amount, (int, float)) or amount <= 0:
        return {
            "reference": "ACC-1036",
            "status": "FAILED",
            "error_code": "INVALID_AMOUNT",
            "message": "Amount must be a positive number",
            "balance_updated": False
        }

    try:
        if account["balance"] < amount:
            raise InsufficientFundsError("Insufficient balance")

        account["balance"] -= amount

        return {
            "reference": "ACC-1036",
            "status": "SUCCESS",
            "error_code": None,
            "message": "Transaction successful",
            "balance_updated": True
        }

    except InsufficientFundsError as e:
        return {
            "reference": "ACC-1036",
            "status": "FAILED",
            "error_code": "INSUFFICIENT_FUNDS",
            "message": str(e),
            "balance_updated": False
        }

### Q37. Validate a nested beneficiary record for banking case 1; use sample reference `ACC-1037` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

def validate_beneficiary(beneficiary):
    reference = "ACC-1037"

    try:
        if not isinstance(beneficiary, dict) or not beneficiary:
            raise InvalidBeneficiaryError("Beneficiary record is empty or invalid")

        name = beneficiary.get("name")
        account_number = beneficiary.get("account_number")
        amount = beneficiary.get("amount")

        if not isinstance(name, str) or not name.strip():
            raise InvalidBeneficiaryError("Invalid beneficiary name")

        if not isinstance(account_number, str) or not account_number.strip():
            raise InvalidBeneficiaryError("Invalid account number")

        if not isinstance(amount, (int, float)) or amount <= 0:
            raise InvalidBeneficiaryError("Invalid transaction amount")

        return {
            "reference": reference,
            "status": "SUCCESS",
            "message": "Beneficiary record is valid"
        }

    except InvalidBeneficiaryError as e:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "INVALID_BENEFICIARY",
            "message": str(e),
            "balance_updated": False
        }

### Q38. Use a context manager for an audit file for banking case 1; use sample reference `ACC-1038` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

def write_audit(reference, message):
    try:
        if not reference or not isinstance(message, str):
            raise AuditFileError("Invalid audit data")

        # Context manager safely opens and closes the file
        with open("audit.log", "a") as file:
            file.write(f"{reference}: {message}\n")

        return {
            "reference": reference,
            "status": "SUCCESS",
            "message": "Audit record written successfully"
        }

    except AuditFileError as e:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "INVALID_AUDIT_DATA",
            "message": str(e)
        }

    except OSError as e:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "AUDIT_FILE_ERROR",
            "message": str(e),
            "cause": type(e).__name__
        }


result = write_audit("ACC-1038", "Transaction completed")
print(result)

{'reference': 'ACC-1038', 'status': 'SUCCESS', 'message': 'Audit record written successfully'}


### Q39. Separate validation and recovery for banking case 1; use sample reference `ACC-1039` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
def process_transaction(account, amount):
    try:
        # 1. Validate
        if not account or not isinstance(amount, (int, float)) or amount <= 0:
            raise ValidationError("Invalid input")

        if account["balance"] < amount:
            raise ValidationError("Insufficient balance")

        # 2. Update balance
        account["balance"] -= amount

        # 3. Success result
        return {
            "reference": "ACC-1039",
            "status": "SUCCESS",
            "balance_updated": True
        }

    # 4. Validation failure
    except ValidationError as e:
        return {
            "reference": "ACC-1039",
            "status": "FAILED",
            "error_code": "VALIDATION_ERROR",
            "message": str(e),
            "balance_updated": False
        }

### Q40. Collect all kyc validation errors for banking case 1; use sample reference `ACC-1040` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
def validate_kyc(kyc):
    errors = []

    if not isinstance(kyc, dict):
        errors.append("Invalid KYC data")
    else:
        if not kyc.get("name"):
            errors.append("Name is required")

        if not kyc.get("id_number"):
            errors.append("ID number is required")

        if not isinstance(kyc.get("age"), int) or kyc["age"] < 18:
            errors.append("Age must be 18 or above")

    if errors:
        return {
            "reference": "ACC-1040",
            "status": "FAILED",
            "error_code": "KYC_VALIDATION_ERROR",
            "errors": errors
        }

    return {
        "reference": "ACC-1040",
        "status": "SUCCESS",
        "message": "KYC validation successful"
    }


# Test
kyc = {"name": "", "id_number": "", "age": 16}
print(validate_kyc(kyc))

{'reference': 'ACC-1040', 'status': 'FAILED', 'error_code': 'KYC_VALIDATION_ERROR', 'errors': ['Name is required', 'ID number is required', 'Age must be 18 or above']}


### Q41. Create insufficientfundserror for banking case 2; use sample reference `ACC-1041` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
def withdraw(balance, amount):
    try:
        if amount > balance:
            raise InsufficientFundsError("Insufficient funds")

        balance = balance - amount

        return {
            "reference": "ACC-1041",
            "status": "SUCCESS",
            "balance": balance
        }

    except InsufficientFundsError as e:
        return {
            "reference": "ACC-1041",
            "status": "FAILED",
            "error_code": "INSUFFICIENT_FUNDS",
            "message": str(e),
            "balance_updated": False
        }


result = withdraw(1000, 1500)
print(result)

NameError: name 'InsufficientFundsError' is not defined

### Q42. Chain invalidtransactionerror from valueerror for banking case 2; use sample reference `ACC-1042` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
class InvalidTransactionError(Exception):
    pass


def process_transaction(amount):
    try:
        if amount <= 0:
            raise ValueError("Amount must be greater than zero")

        return {
            "reference": "ACC-1042",
            "status": "SUCCESS",
            "amount": amount
        }

    except ValueError as e:
        raise InvalidTransactionError("Invalid transaction") from e


try:
    result = process_transaction(-100)
    print(result)

except InvalidTransactionError as e:
    print({
        "reference": "ACC-1042",
        "status": "FAILED",
        "error_code": "INVALID_TRANSACTION",
        "message": str(e),
        "cause": str(e.__cause__)
    })

{'reference': 'ACC-1042', 'status': 'FAILED', 'error_code': 'INVALID_TRANSACTION', 'message': 'Invalid transaction', 'cause': 'Amount must be greater than zero'}


### Q43. Process transfers while isolating bad rows for banking case 2; use sample reference `ACC-1043` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class InvalidTransferError(Exception):
    pass


def process_transfers(transfers):
    results = []

    for row in transfers:
        try:
            if not isinstance(row, dict):
                raise InvalidTransferError("Invalid transfer row")

            amount = row.get("amount")

            if not isinstance(amount, (int, float)) or amount <= 0:
                raise InvalidTransferError("Invalid transfer amount")

            results.append({
                "reference": "ACC-1043",
                "status": "SUCCESS",
                "amount": amount
            })

        except InvalidTransferError as e:
            results.append({
                "reference": "ACC-1043",
                "status": "FAILED",
                "error_code": "INVALID_TRANSFER",
                "message": str(e)
            })

    return results


transfers = [
    {"amount": 500},
    {"amount": -100},
    {"amount": 200}
]

print(process_transfers(transfers))

[{'reference': 'ACC-1043', 'status': 'SUCCESS', 'amount': 500}, {'reference': 'ACC-1043', 'status': 'FAILED', 'error_code': 'INVALID_TRANSFER', 'message': 'Invalid transfer amount'}, {'reference': 'ACC-1043', 'status': 'SUCCESS', 'amount': 200}]


### Q44. Retry a transient payment gateway for banking case 2; use sample reference `ACC-1044` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class PaymentGatewayError(Exception):
    pass


def process_payment(amount, attempts=3):
    reference = "ACC-1044"

    if not isinstance(amount, (int, float)) or amount <= 0:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "INVALID_AMOUNT",
            "message": "Invalid payment amount"
        }

    for attempt in range(attempts):
        try:
            # Simulate payment gateway
            if attempt < 2:
                raise PaymentGatewayError("Gateway temporarily unavailable")

            return {
                "reference": reference,
                "status": "SUCCESS",
                "message": "Payment successful",
                "attempts": attempt + 1
            }

        except PaymentGatewayError as e:
            if attempt == attempts - 1:
                return {
                    "reference": reference,
                    "status": "FAILED",
                    "error_code": "GATEWAY_ERROR",
                    "message": str(e),
                    "balance_updated": False
                }


print(process_payment(500))

{'reference': 'ACC-1044', 'status': 'SUCCESS', 'message': 'Payment successful', 'attempts': 3}


### Q45. Log failed standing instructions for banking case 2; use sample reference `ACC-1045` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class StandingInstructionError(Exception):
    pass


def process_standing_instruction(instruction):
    reference = "ACC-1045"

    try:
        if not isinstance(instruction, dict) or not instruction:
            raise StandingInstructionError("Invalid instruction")

        amount = instruction.get("amount")

        if not isinstance(amount, (int, float)) or amount <= 0:
            raise StandingInstructionError("Invalid amount")

        # Simulate a failed standing instruction
        if instruction.get("status") == "FAILED":
            raise StandingInstructionError("Standing instruction failed")

        return {
            "reference": reference,
            "status": "SUCCESS",
            "message": "Standing instruction completed"
        }

    except StandingInstructionError as e:
        print(f"LOG: {reference} - {e}")

        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "STANDING_INSTRUCTION_ERROR",
            "message": str(e),
            "balance_updated": False
        }


instruction = {
    "amount": 500,
    "status": "FAILED"
}

print(process_standing_instruction(instruction))

LOG: ACC-1045 - Standing instruction failed
{'reference': 'ACC-1045', 'status': 'FAILED', 'error_code': 'STANDING_INSTRUCTION_ERROR', 'message': 'Standing instruction failed', 'balance_updated': False}


### Q46. Return a structured failure result for banking case 2; use sample reference `ACC-1046` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


def process_transaction(balance, amount):
    reference = "ACC-1046"

    try:
        if not isinstance(amount, (int, float)) or amount <= 0:
            raise BankingError("Invalid transaction amount")

        if amount > balance:
            raise BankingError("Insufficient funds")

        balance -= amount

        return {
            "reference": reference,
            "status": "SUCCESS",
            "balance": balance
        }

    except BankingError as e:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "TRANSACTION_FAILED",
            "message": str(e),
            "balance_updated": False
        }


print(process_transaction(1000, 1500))

{'reference': 'ACC-1046', 'status': 'FAILED', 'error_code': 'TRANSACTION_FAILED', 'message': 'Insufficient funds', 'balance_updated': False}


### Q47. Validate a nested beneficiary record for banking case 2; use sample reference `ACC-1047` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BeneficiaryValidationError(Exception):
    pass


def validate_beneficiary(data):
    reference = "ACC-1047"

    try:
        if not isinstance(data, dict) or not data:
            raise BeneficiaryValidationError("Invalid beneficiary data")

        beneficiary = data.get("beneficiary")

        if not isinstance(beneficiary, dict):
            raise BeneficiaryValidationError("Beneficiary must be a dictionary")

        if not beneficiary.get("name"):
            raise BeneficiaryValidationError("Beneficiary name is required")

        if not beneficiary.get("account_number"):
            raise BeneficiaryValidationError("Account number is required")

        return {
            "reference": reference,
            "status": "SUCCESS",
            "message": "Beneficiary is valid"
        }

    except BeneficiaryValidationError as e:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "INVALID_BENEFICIARY",
            "message": str(e),
            "balance_updated": False
        }


data = {
    "beneficiary": {
        "name": "",
        "account_number": "123456"
    }
}

print(validate_beneficiary(data))

{'reference': 'ACC-1047', 'status': 'FAILED', 'error_code': 'INVALID_BENEFICIARY', 'message': 'Beneficiary name is required', 'balance_updated': False}


### Q48. Use a context manager for an audit file for banking case 2; use sample reference `ACC-1048` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

def write_audit(message):
    reference = "ACC-1048"

    try:
        if not isinstance(message, str) or not message:
            raise ValueError("Invalid audit message")

        # Context manager safely opens and closes the file
        with open("audit.log", "a") as file:
            file.write(f"{reference}: {message}\n")

        return {
            "reference": reference,
            "status": "SUCCESS",
            "message": "Audit record written"
        }

    except ValueError as e:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "INVALID_AUDIT_DATA",
            "message": str(e)
        }

    except OSError as e:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "AUDIT_FILE_ERROR",
            "message": str(e),
            "cause": str(e)
        }


print(write_audit("Transaction failed"))

{'reference': 'ACC-1048', 'status': 'SUCCESS', 'message': 'Audit record written'}


### Q49. Separate validation and recovery for banking case 2; use sample reference `ACC-1049` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class ValidationError(Exception):
    pass


def validate(amount, balance):
    if not isinstance(amount, (int, float)) or amount <= 0:
        raise ValidationError("Invalid amount")

    if amount > balance:
        raise ValidationError("Insufficient funds")


def process_transaction(balance, amount):
    reference = "ACC-1049"

    try:
        # Step 1: Validation
        validate(amount, balance)

        # Step 2: Update only after validation succeeds
        balance -= amount

        return {
            "reference": reference,
            "status": "SUCCESS",
            "balance": balance,
            "balance_updated": True
        }

    except ValidationError as e:
        # Step 3: Recovery
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "VALIDATION_ERROR",
            "message": str(e),
            "balance_updated": False
        }


print(process_transaction(1000, 1500))

{'reference': 'ACC-1049', 'status': 'FAILED', 'error_code': 'VALIDATION_ERROR', 'message': 'Insufficient funds', 'balance_updated': False}


### Q50. Collect all kyc validation errors for banking case 2; use sample reference `ACC-1050` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

def validate_kyc(data):
    errors = []

    if not isinstance(data, dict):
        errors.append("KYC data must be a dictionary")
        return errors

    if not data.get("name"):
        errors.append("Name is required")

    if not data.get("id_number"):
        errors.append("ID number is required")

    if "age" not in data:
        errors.append("Age is required")
    elif not isinstance(data["age"], int):
        errors.append("Age must be an integer")
    elif data["age"] < 18:
        errors.append("Age must be 18 or above")

    return errors


def process_kyc(data):
    errors = validate_kyc(data)

    if errors:
        return {
            "reference": "ACC-1050",
            "status": "FAILED",
            "error_code": "KYC_VALIDATION_ERROR",
            "errors": errors,
            "balance_updated": False
        }

    return {
        "reference": "ACC-1050",
        "status": "SUCCESS",
        "message": "KYC validation successful"
    }


# Test
kyc_data = {
    "name": "",
    "id_number": "",
    "age": 16
}

print(process_kyc(kyc_data))

{'reference': 'ACC-1050', 'status': 'FAILED', 'error_code': 'KYC_VALIDATION_ERROR', 'errors': ['Name is required', 'ID number is required', 'Age must be 18 or above'], 'balance_updated': False}


### Q51. Create insufficientfundserror for banking case 3; use sample reference `ACC-1051` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class InsufficientFundsError(Exception):
    pass


def withdraw(balance, amount):
    reference = "ACC-1051"

    try:
        if not isinstance(amount, (int, float)) or amount <= 0:
            raise ValueError("Invalid amount")

        if amount > balance:
            raise InsufficientFundsError("Insufficient funds")

        new_balance = balance - amount

        return {
            "reference": reference,
            "status": "SUCCESS",
            "balance": new_balance
        }

    except InsufficientFundsError as e:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "INSUFFICIENT_FUNDS",
            "message": str(e),
            "balance_updated": False
        }


print(withdraw(1000, 1500))

{'reference': 'ACC-1051', 'status': 'FAILED', 'error_code': 'INSUFFICIENT_FUNDS', 'message': 'Insufficient funds', 'balance_updated': False}


### Q52. Chain invalidtransactionerror from valueerror for banking case 3; use sample reference `ACC-1052` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class InvalidTransactionError(Exception):
    pass


def process_transaction(amount):
    try:
        if not isinstance(amount, (int, float)):
            raise ValueError("Amount must be a number")

        if amount <= 0:
            raise ValueError("Amount must be greater than zero")

        return {
            "reference": "ACC-1052",
            "status": "SUCCESS",
            "amount": amount
        }

    except ValueError as e:
        raise InvalidTransactionError("Invalid transaction") from e


try:
    print(process_transaction(-100))

except InvalidTransactionError as e:
    print({
        "reference": "ACC-1052",
        "status": "FAILED",
        "error_code": "INVALID_TRANSACTION",
        "message": str(e),
        "cause": str(e.__cause__),
        "balance_updated": False
    })

{'reference': 'ACC-1052', 'status': 'FAILED', 'error_code': 'INVALID_TRANSACTION', 'message': 'Invalid transaction', 'cause': 'Amount must be greater than zero', 'balance_updated': False}


### Q53. Process transfers while isolating bad rows for banking case 3; use sample reference `ACC-1053` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class InvalidTransferError(Exception):
    pass


def process_transfers(transfers):
    results = []

    for row in transfers:
        try:
            if not isinstance(row, dict):
                raise InvalidTransferError("Invalid transfer row")

            amount = row.get("amount")

            if not isinstance(amount, (int, float)) or amount <= 0:
                raise InvalidTransferError("Invalid transfer amount")

            results.append({
                "reference": "ACC-1053",
                "status": "SUCCESS",
                "amount": amount
            })

        except InvalidTransferError as e:
            results.append({
                "reference": "ACC-1053",
                "status": "FAILED",
                "error_code": "INVALID_TRANSFER",
                "message": str(e)
            })

    return results


transfers = [
    {"amount": 500},
    {"amount": -100},
    {"amount": 200}
]

print(process_transfers(transfers))

[{'reference': 'ACC-1053', 'status': 'SUCCESS', 'amount': 500}, {'reference': 'ACC-1053', 'status': 'FAILED', 'error_code': 'INVALID_TRANSFER', 'message': 'Invalid transfer amount'}, {'reference': 'ACC-1053', 'status': 'SUCCESS', 'amount': 200}]


### Q54. Retry a transient payment gateway for banking case 3; use sample reference `ACC-1054` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class PaymentGatewayError(Exception):
    pass


def process_payment(amount):
    reference = "ACC-1054"

    if not isinstance(amount, (int, float)) or amount <= 0:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "INVALID_AMOUNT",
            "message": "Invalid payment amount"
        }

    for attempt in range(3):
        try:
            # Simulate temporary gateway failure
            if attempt < 2:
                raise PaymentGatewayError("Gateway temporarily unavailable")

            return {
                "reference": reference,
                "status": "SUCCESS",
                "message": "Payment successful",
                "attempts": attempt + 1
            }

        except PaymentGatewayError as e:
            if attempt == 2:
                return {
                    "reference": reference,
                    "status": "FAILED",
                    "error_code": "GATEWAY_ERROR",
                    "message": str(e),
                    "balance_updated": False
                }


print(process_payment(500))

{'reference': 'ACC-1054', 'status': 'SUCCESS', 'message': 'Payment successful', 'attempts': 3}


### Q55. Log failed standing instructions for banking case 3; use sample reference `ACC-1055` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class StandingInstructionError(Exception):
    pass


def process_standing_instruction(instruction):
    reference = "ACC-1055"

    try:
        if not isinstance(instruction, dict) or not instruction:
            raise StandingInstructionError("Invalid instruction")

        amount = instruction.get("amount")

        if not isinstance(amount, (int, float)) or amount <= 0:
            raise StandingInstructionError("Invalid amount")

        if instruction.get("status") == "FAILED":
            raise StandingInstructionError("Standing instruction failed")

        return {
            "reference": reference,
            "status": "SUCCESS",
            "message": "Standing instruction completed"
        }

    except StandingInstructionError as e:
        print(f"LOG: {reference} - {e}")

        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "STANDING_INSTRUCTION_ERROR",
            "message": str(e),
            "balance_updated": False
        }


instruction = {
    "amount": 500,
    "status": "FAILED"
}

print(process_standing_instruction(instruction))

LOG: ACC-1055 - Standing instruction failed
{'reference': 'ACC-1055', 'status': 'FAILED', 'error_code': 'STANDING_INSTRUCTION_ERROR', 'message': 'Standing instruction failed', 'balance_updated': False}


### Q56. Return a structured failure result for banking case 3; use sample reference `ACC-1056` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


def process_transaction(balance, amount):
    reference = "ACC-1056"

    try:
        if not isinstance(amount, (int, float)) or amount <= 0:
            raise BankingError("Invalid transaction amount")

        if amount > balance:
            raise BankingError("Insufficient funds")

        new_balance = balance - amount

        return {
            "reference": reference,
            "status": "SUCCESS",
            "balance": new_balance,
            "balance_updated": True
        }

    except BankingError as e:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "TRANSACTION_FAILED",
            "message": str(e),
            "balance_updated": False
        }


print(process_transaction(1000, 1500))

{'reference': 'ACC-1056', 'status': 'FAILED', 'error_code': 'TRANSACTION_FAILED', 'message': 'Insufficient funds', 'balance_updated': False}


### Q57. Validate a nested beneficiary record for banking case 3; use sample reference `ACC-1057` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BeneficiaryValidationError(Exception):
    pass


def validate_beneficiary(data):
    reference = "ACC-1057"

    try:
        if not isinstance(data, dict) or not data:
            raise BeneficiaryValidationError("Invalid beneficiary data")

        beneficiary = data.get("beneficiary")

        if not isinstance(beneficiary, dict):
            raise BeneficiaryValidationError("Beneficiary must be a dictionary")

        if not beneficiary.get("name"):
            raise BeneficiaryValidationError("Beneficiary name is required")

        if not beneficiary.get("account_number"):
            raise BeneficiaryValidationError("Account number is required")

        return {
            "reference": reference,
            "status": "SUCCESS",
            "message": "Beneficiary is valid"
        }

    except BeneficiaryValidationError as e:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "INVALID_BENEFICIARY",
            "message": str(e),
            "balance_updated": False
        }


data = {
    "beneficiary": {
        "name": "",
        "account_number": "123456"
    }
}

print(validate_beneficiary(data))

{'reference': 'ACC-1057', 'status': 'FAILED', 'error_code': 'INVALID_BENEFICIARY', 'message': 'Beneficiary name is required', 'balance_updated': False}


### Q58. Use a context manager for an audit file for banking case 3; use sample reference `ACC-1058` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

def write_audit(message):
    reference = "ACC-1058"

    try:
        if not isinstance(message, str) or not message:
            raise ValueError("Invalid audit message")

        # Context manager automatically closes the file
        with open("audit.log", "a") as file:
            file.write(f"{reference}: {message}\n")

        return {
            "reference": reference,
            "status": "SUCCESS",
            "message": "Audit record written successfully"
        }

    except ValueError as e:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "INVALID_AUDIT_DATA",
            "message": str(e)
        }

    except OSError as e:
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "AUDIT_FILE_ERROR",
            "message": str(e),
            "cause": str(e)
        }


print(write_audit("Transaction failed"))

{'reference': 'ACC-1058', 'status': 'SUCCESS', 'message': 'Audit record written successfully'}


### Q59. Separate validation and recovery for banking case 3; use sample reference `ACC-1059` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class ValidationError(Exception):
    pass


def validate(balance, amount):
    if not isinstance(amount, (int, float)) or amount <= 0:
        raise ValidationError("Invalid amount")

    if amount > balance:
        raise ValidationError("Insufficient funds")


def process_transaction(balance, amount):
    reference = "ACC-1059"

    try:
        # Step 1: Validate
        validate(balance, amount)

        # Step 2: Update balance only after validation
        new_balance = balance - amount

        return {
            "reference": reference,
            "status": "SUCCESS",
            "balance": new_balance,
            "balance_updated": True
        }

    except ValidationError as e:
        # Step 3: Recovery
        return {
            "reference": reference,
            "status": "FAILED",
            "error_code": "VALIDATION_ERROR",
            "message": str(e),
            "balance_updated": False
        }


print(process_transaction(1000, 1500))

{'reference': 'ACC-1059', 'status': 'FAILED', 'error_code': 'VALIDATION_ERROR', 'message': 'Insufficient funds', 'balance_updated': False}


### Q60. Collect all kyc validation errors for banking case 3; use sample reference `ACC-1060` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

def validate_kyc(data):
    errors = []

    if not isinstance(data, dict):
        errors.append("KYC data must be a dictionary")
        return errors

    if not data.get("name"):
        errors.append("Name is required")

    if not data.get("id_number"):
        errors.append("ID number is required")

    if "age" not in data:
        errors.append("Age is required")
    elif not isinstance(data["age"], int):
        errors.append("Age must be an integer")
    elif data["age"] < 18:
        errors.append("Age must be 18 or above")

    return errors


def process_kyc(data):
    errors = validate_kyc(data)

    if errors:
        return {
            "reference": "ACC-1060",
            "status": "FAILED",
            "error_code": "KYC_VALIDATION_ERROR",
            "errors": errors,
            "balance_updated": False
        }

    return {
        "reference": "ACC-1060",
        "status": "SUCCESS",
        "message": "KYC validation successful"
    }


kyc_data = {
    "name": "",
    "id_number": "",
    "age": 16
}

print(process_kyc(kyc_data))

{'reference': 'ACC-1060', 'status': 'FAILED', 'error_code': 'KYC_VALIDATION_ERROR', 'errors': ['Name is required', 'ID number is required', 'Age must be 18 or above'], 'balance_updated': False}


### Q61. Create insufficientfundserror for banking case 4; use sample reference `ACC-1061` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class InsufficientFundsError(Exception):
    """Raised when an account lacks sufficient funds for a withdrawal."""
    pass


def withdraw(account, amount):
    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise TypeError("amount must be a number")

    if amount < 0:
        raise ValueError("amount cannot be negative")

    balance = account["balance"]

    if amount > balance:
        raise InsufficientFundsError(
            f"Insufficient funds in {account['id']}: "
            f"requested={amount}, available={balance}"
        )

    # Update only after all business-rule checks pass.
    account["balance"] = balance - amount
    return account["balance"]

### Q62. Chain invalidtransactionerror from valueerror for banking case 4; use sample reference `ACC-1062` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class InvalidTransactionError(Exception):
    pass


def withdraw(account, amount):
    try:
        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise ValueError("Invalid transaction amount")

        if amount <= 0:
            raise ValueError("Transaction amount must be positive")

        balance = account["balance"]

        if amount > balance:
            raise ValueError("Insufficient funds")

    except ValueError as exc:
        raise InvalidTransactionError(
            f"Invalid transaction for account {account.get('id', 'unknown')}"
        ) from exc

    account["balance"] = balance - amount
    return account["balance"]


# Sample reference: ACC-1062
account = {"id": "ACC-1062", "balance": 5000}

try:
    withdraw(account, -100)
except InvalidTransactionError as e:
    print(e)
    print("Cause:", e.__cause__)

print("Balance:", account["balance"])

Invalid transaction for account ACC-1062
Cause: Transaction amount must be positive
Balance: 5000


### Q63. Process transfers while isolating bad rows for banking case 4; use sample reference `ACC-1063` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class InvalidTransactionError(Exception):
    pass


def process_transfers(account, transfers):
    if not isinstance(transfers, list):
        raise TypeError("transfers must be a list")

    results = []

    for row in transfers:
        try:
            if not isinstance(row, dict):
                raise ValueError("Invalid transfer row")

            amount = row["amount"]

            if not isinstance(amount, (int, float)) or isinstance(amount, bool):
                raise ValueError("Invalid transfer amount")

            if amount <= 0:
                raise ValueError("Transfer amount must be positive")

            if amount > account["balance"]:
                raise ValueError("Insufficient funds")

            # Update only after the complete row is validated.
            account["balance"] -= amount
            results.append({
                "status": "success",
                "amount": amount
            })

        except (ValueError, KeyError) as exc:
            # Isolate the bad row and preserve the original cause.
            results.append({
                "status": "failed",
                "error": InvalidTransactionError(
                    f"Invalid transfer for {account.get('id', 'unknown')}"
                ),
                "cause": exc
            })

    return results


# Sample reference: ACC-1063
account = {"id": "ACC-1063", "balance": 5000}

transfers = [
    {"amount": 1000},     # valid
    {"amount": 6000},     # insufficient funds -> isolated
    {"amount": 500},      # valid
    {"amount": "abc"},    # invalid type -> isolated
]

results = process_transfers(account, transfers)

print(results)
print("Final balance:", account["balance"])

[{'status': 'success', 'amount': 1000}, {'status': 'failed', 'error': InvalidTransactionError('Invalid transfer for ACC-1063'), 'cause': ValueError('Insufficient funds')}, {'status': 'success', 'amount': 500}, {'status': 'failed', 'error': InvalidTransactionError('Invalid transfer for ACC-1063'), 'cause': ValueError('Invalid transfer amount')}]
Final balance: 3500


### Q64. Retry a transient payment gateway for banking case 4; use sample reference `ACC-1064` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class PaymentGatewayError(Exception):
    pass


def process_payment(account, amount, gateway, retries=3):
    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise TypeError("amount must be a number")

    if amount <= 0:
        raise ValueError("amount must be positive")

    if amount > account["balance"]:
        raise ValueError("Insufficient funds")

    last_error = None

    for attempt in range(retries):
        try:
            # Smallest risky statement
            gateway.charge(account["id"], amount)

            # Update balance only after successful payment
            account["balance"] -= amount
            return "Payment successful"

        except PaymentGatewayError as exc:
            last_error = exc

    raise PaymentGatewayError(
        f"Payment failed after {retries} attempts for {account['id']}"
    ) from last_error


# Sample reference: ACC-1064
class Gateway:
    def __init__(self):
        self.calls = 0

    def charge(self, account_id, amount):
        self.calls += 1

        if self.calls < 3:
            raise PaymentGatewayError("Temporary gateway failure")

        return True


account = {"id": "ACC-1064", "balance": 5000}
gateway = Gateway()

result = process_payment(account, 1000, gateway)

print(result)
print("Balance:", account["balance"])

Payment successful
Balance: 4000


### Q65. Log failed standing instructions for banking case 4; use sample reference `ACC-1065` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class StandingInstructionError(Exception):
    pass


def process_standing_instruction(account, amount, logger):
    try:
        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise ValueError("Invalid amount type")

        if amount <= 0:
            raise ValueError("Amount must be positive")

        if amount > account["balance"]:
            raise ValueError("Insufficient funds")

        # Update balance only after validation succeeds
        account["balance"] -= amount
        return "Standing instruction successful"

    except ValueError as exc:
        error = StandingInstructionError(
            f"Standing instruction failed for {account.get('id', 'unknown')}"
        )

        # Log the failed instruction
        logger.append({
            "account": account.get("id", "unknown"),
            "amount": amount,
            "error": str(error),
            "cause": str(exc)
        })

        raise error from exc


# Sample reference: ACC-1065
account = {"id": "ACC-1065", "balance": 5000}
logs = []

try:
    process_standing_instruction(account, 6000, logs)
except StandingInstructionError as exc:
    print(exc)
    print("Cause:", exc.__cause__)

print("Balance:", account["balance"])
print("Logs:", logs)

Standing instruction failed for ACC-1065
Cause: Insufficient funds
Balance: 5000
Logs: [{'account': 'ACC-1065', 'amount': 6000, 'error': 'Standing instruction failed for ACC-1065', 'cause': 'Insufficient funds'}]


### Q66. Return a structured failure result for banking case 4; use sample reference `ACC-1066` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
class InvalidTransactionError(Exception):
    pass

def transfer(account, amount):
    try:
        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise ValueError("Invalid amount type")

        if amount <= 0:
            raise ValueError("Amount must be positive")

        if amount > account["balance"]:
            raise ValueError("Insufficient funds")

        account["balance"] -= amount

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None,
            "cause": None
        }

    except ValueError as exc:
        error = InvalidTransactionError(
            f"Transaction failed for {account.get('id', 'unknown')}"
        )

        return {
            "success": False,
            "account": account.get("id", "unknown"),
            "amount": amount,
            "balance": account.get("balance"),
            "error": str(error),
            "cause": str(exc)
        }


account = {
    "id": "ACC-1066",
    "balance": 5000
}

result = transfer(account, 6000)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1066', 'amount': 6000, 'balance': 5000, 'error': 'Transaction failed for ACC-1066', 'cause': 'Insufficient funds'}
Balance: 5000


### Q67. Validate a nested beneficiary record for banking case 4; use sample reference `ACC-1067` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class InvalidBeneficiaryError(Exception):
    pass


def validate_beneficiary(account):
    try:
        beneficiary = account["beneficiary"]

        if not isinstance(beneficiary, dict):
            raise ValueError("Beneficiary must be a dictionary")

        name = beneficiary["name"]
        account_number = beneficiary["account_number"]

        if not isinstance(name, str) or not name.strip():
            raise ValueError("Invalid beneficiary name")

        if not isinstance(account_number, str) or not account_number.strip():
            raise ValueError("Invalid beneficiary account number")

        return {
            "success": True,
            "account": account["id"],
            "beneficiary": beneficiary,
            "error": None,
            "cause": None
        }

    except (KeyError, ValueError) as exc:
        error = InvalidBeneficiaryError(
            f"Beneficiary validation failed for "
            f"{account.get('id', 'unknown')}"
        )

        return {
            "success": False,
            "account": account.get("id", "unknown"),
            "beneficiary": account.get("beneficiary"),
            "balance": account.get("balance"),
            "error": str(error),
            "cause": str(exc)
        }


# Sample reference: ACC-1067
account = {
    "id": "ACC-1067",
    "balance": 5000,
    "beneficiary": {
        "name": "",
        "account_number": "BEN-1001"
    }
}

result = validate_beneficiary(account)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1067', 'beneficiary': {'name': '', 'account_number': 'BEN-1001'}, 'balance': 5000, 'error': 'Beneficiary validation failed for ACC-1067', 'cause': 'Invalid beneficiary name'}
Balance: 5000


### Q68. Use a context manager for an audit file for banking case 4; use sample reference `ACC-1068` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class AuditError(Exception):
    pass


def audit_transaction(account, amount, filename):
    try:
        if not isinstance(account, dict):
            raise ValueError("Invalid account")

        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise ValueError("Invalid amount type")

        if amount <= 0:
            raise ValueError("Amount must be positive")

        if amount > account["balance"]:
            raise ValueError("Insufficient funds")

        # Audit before updating the balance
        with open(filename, "a") as file:
            file.write(
                f"Account: {account['id']}, "
                f"Amount: {amount}\n"
            )

        # Update only after validation and audit succeed
        account["balance"] -= amount

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None,
            "cause": None
        }

    except (ValueError, OSError) as exc:
        error = AuditError(
            f"Transaction failed for {account.get('id', 'unknown')}"
            if isinstance(account, dict)
            else "Transaction failed for unknown"
        )

        return {
            "success": False,
            "account": account.get("id", "unknown")
            if isinstance(account, dict)
            else "unknown",
            "amount": amount,
            "balance": account.get("balance")
            if isinstance(account, dict)
            else None,
            "error": str(error),
            "cause": str(exc)
        }


# Sample reference: ACC-1068
account = {
    "id": "ACC-1068",
    "balance": 5000
}

result = audit_transaction(account, 6000, "audit.txt")

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1068', 'amount': 6000, 'balance': 5000, 'error': 'Transaction failed for ACC-1068', 'cause': 'Insufficient funds'}
Balance: 5000


### Q69. Separate validation and recovery for banking case 4; use sample reference `ACC-1069` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Write your solution here


### Q70. Collect all kyc validation errors for banking case 4; use sample reference `ACC-1070` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class InvalidTransactionError(Exception):
    pass


def validate_transfer(account, amount):
    if not isinstance(account, dict):
        raise ValueError("Invalid account")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValueError("Invalid amount type")

    if amount <= 0:
        raise ValueError("Amount must be positive")

    if "balance" not in account:
        raise KeyError("Missing balance")

    if amount > account["balance"]:
        raise ValueError("Insufficient funds")

    return True


def transfer_with_recovery(account, amount):
    try:
        # Validation is completed before any balance update.
        validate_transfer(account, amount)

        account["balance"] -= amount

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None,
            "cause": None
        }

    except (ValueError, KeyError) as exc:
        error = InvalidTransactionError(
            f"Transaction failed for "
            f"{account.get('id', 'unknown')}"
        )

        return {
            "success": False,
            "account": account.get("id", "unknown"),
            "amount": amount,
            "balance": account.get("balance"),
            "error": str(error),
            "cause": str(exc)
        }


# Sample reference: ACC-1069
account = {
    "id": "ACC-1069",
    "balance": 5000
}

result = transfer_with_recovery(account, 6000)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1069', 'amount': 6000, 'balance': 5000, 'error': 'Transaction failed for ACC-1069', 'cause': 'Insufficient funds'}
Balance: 5000



---

## Difficult

Architecture, edge cases, and end-to-end banking systems.


### Q71. Design a banking exception hierarchy for banking case 1; use sample reference `ACC-1071` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    """Base exception for banking operations."""
    pass


class ValidationError(BankingError):
    """Invalid input or transaction data."""
    pass


class InsufficientFundsError(BankingError):
    """Transaction exceeds the available balance."""
    pass


class AccountError(BankingError):
    """Invalid or incomplete account information."""
    pass


def transfer(account, amount):
    try:
        # Validate account
        if not isinstance(account, dict):
            raise AccountError("Invalid account")

        if "id" not in account or "balance" not in account:
            raise AccountError("Missing account information")

        # Validate amount
        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise ValidationError("Invalid amount type")

        if amount <= 0:
            raise ValidationError("Amount must be positive")

        # Business-rule validation
        if amount > account["balance"]:
            raise InsufficientFundsError("Insufficient funds")

        # Update only after all validation succeeds
        account["balance"] -= amount

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None,
            "cause": None
        }

    except BankingError as exc:
        return {
            "success": False,
            "account": account.get("id", "unknown")
            if isinstance(account, dict) else "unknown",
            "amount": amount,
            "balance": account.get("balance")
            if isinstance(account, dict) else None,
            "error": str(exc),
            "cause": type(exc).__name__
        }


# Sample reference: ACC-1071
account = {
    "id": "ACC-1071",
    "balance": 5000
}

result = transfer(account, 6000)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1071', 'amount': 6000, 'balance': 5000, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError'}
Balance: 5000


### Q72. Make a transfer rollback-safe for banking case 1; use sample reference `ACC-1072` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


def transfer(account, amount):
    original_balance = None

    try:
        # Validate account
        if not isinstance(account, dict):
            raise ValidationError("Invalid account")

        if "id" not in account or "balance" not in account:
            raise ValidationError("Missing account information")

        original_balance = account["balance"]

        # Validate amount
        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise ValidationError("Invalid amount type")

        if amount <= 0:
            raise ValidationError("Amount must be positive")

        if amount > account["balance"]:
            raise InsufficientFundsError("Insufficient funds")

        # Transaction update
        account["balance"] -= amount

        # Return successful result
        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None,
            "cause": None
        }

    except BankingError as exc:
        # Roll back any balance change
        if original_balance is not None:
            account["balance"] = original_balance

        return {
            "success": False,
            "account": account.get("id", "unknown")
            if isinstance(account, dict) else "unknown",
            "amount": amount,
            "balance": account.get("balance")
            if isinstance(account, dict) else None,
            "error": str(exc),
            "cause": type(exc).__name__
        }


# Sample reference: ACC-1072
account = {
    "id": "ACC-1072",
    "balance": 5000
}

result = transfer(account, 6000)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1072', 'amount': 6000, 'balance': 5000, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError'}
Balance: 5000


### Q73. Preserve causes across service layers for banking case 1; use sample reference `ACC-1073` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class TransferServiceError(BankingError):
    pass


def validate_transfer(account, amount):
    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")


def account_service(account, amount):
    original_balance = account.get("balance")

    try:
        validate_transfer(account, amount)

        account["balance"] -= amount

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"]
        }

    except BankingError:
        # No partial update should remain after a banking failure.
        if original_balance is not None:
            account["balance"] = original_balance
        raise


def transfer_service(account, amount):
    try:
        return account_service(account, amount)

    except BankingError as exc:
        # Wrap the lower-level error but preserve its cause.
        raise TransferServiceError(
            f"Transfer service failed for "
            f"{account.get('id', 'unknown')}"
        ) from exc


def process_transfer(account, amount):
    try:
        return transfer_service(account, amount)

    except TransferServiceError as exc:
        return {
            "success": False,
            "account": account.get("id", "unknown"),
            "amount": amount,
            "balance": account.get("balance"),
            "error": str(exc),
            "cause": str(exc.__cause__),
            "cause_type": type(exc.__cause__).__name__
        }


# Sample reference: ACC-1073
account = {
    "id": "ACC-1073",
    "balance": 5000
}

result = process_transfer(account, 6000)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1073', 'amount': 6000, 'balance': 5000, 'error': 'Transfer service failed for ACC-1073', 'cause': 'Insufficient funds', 'cause_type': 'InsufficientFundsError'}
Balance: 5000


### Q74. Implement bounded exponential retry for banking case 1; use sample reference `ACC-1074` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

import time


class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class ServiceUnavailableError(BankingError):
    pass


def debit_account(account, amount):
    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")

    # Smallest risky operation: the actual account update.
    account["balance"] -= amount

    return {
        "success": True,
        "account": account["id"],
        "amount": amount,
        "balance": account["balance"]
    }


def transfer_with_retry(account, amount, max_retries=3, base_delay=0.1):
    original_balance = account.get("balance") if isinstance(account, dict) else None

    for attempt in range(max_retries + 1):
        try:
            return debit_account(account, amount)

        except (ValidationError, InsufficientFundsError):
            # Business-rule failures are not retryable.
            if original_balance is not None:
                account["balance"] = original_balance
            raise

        except ServiceUnavailableError as exc:
            # Retry only the temporary infrastructure failure.
            if attempt == max_retries:
                if original_balance is not None:
                    account["balance"] = original_balance

                raise BankingError(
                    f"Transfer failed after {max_retries + 1} attempts"
                ) from exc

            delay = base_delay * (2 ** attempt)
            time.sleep(delay)


# Sample reference: ACC-1074
account = {
    "id": "ACC-1074",
    "balance": 5000
}

try:
    result = transfer_with_retry(account, 6000)
    print(result)
except BankingError as exc:
    print({
        "success": False,
        "account": account["id"],
        "amount": 6000,
        "balance": account["balance"],
        "error": str(exc),
        "cause": str(exc.__cause__) if exc.__cause__ else None
    })

print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1074', 'amount': 6000, 'balance': 5000, 'error': 'Insufficient funds', 'cause': None}
Balance: 5000


### Q75. Create a transaction context manager for banking case 1; use sample reference `ACC-1075` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

from contextlib import contextmanager


class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


@contextmanager
def transaction(account):
    original_balance = account.get("balance")

    try:
        yield
    except BankingError:
        # Roll back any partial balance update.
        if original_balance is not None:
            account["balance"] = original_balance
        raise


def transfer(account, amount):
    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")

    with transaction(account):
        # Update happens inside the transaction.
        account["balance"] -= amount

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None
        }


# Sample reference: ACC-1075
account = {
    "id": "ACC-1075",
    "balance": 5000
}

try:
    result = transfer(account, 6000)
except BankingError as exc:
    result = {
        "success": False,
        "account": account.get("id", "unknown"),
        "amount": 6000,
        "balance": account.get("balance"),
        "error": str(exc),
        "cause": type(exc).__name__
    }

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1075', 'amount': 6000, 'balance': 5000, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError'}
Balance: 5000


### Q76. Build a dead-letter queue for payment events for banking case 1; use sample reference `ACC-1076` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

from collections import deque


class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class PaymentProcessingError(BankingError):
    pass


dead_letter_queue = deque()


def process_payment(event):
    if not isinstance(event, dict):
        raise ValidationError("Event must be a dictionary")

    if "account" not in event or "amount" not in event:
        raise ValidationError("Missing payment information")

    account = event["account"]
    amount = event["amount"]

    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")

    # Smallest risky state-changing statement.
    account["balance"] -= amount

    return {
        "success": True,
        "account": account["id"],
        "amount": amount,
        "balance": account["balance"]
    }


def handle_payment(event, max_attempts=3):
    if not isinstance(event, dict):
        dead_letter_queue.append({
            "event": event,
            "error": "Event must be a dictionary",
            "cause": "ValidationError",
            "attempts": 1
        })
        return {
            "success": False,
            "queued": True
        }

    account = event.get("account")

    if not isinstance(account, dict):
        dead_letter_queue.append({
            "event": event,
            "error": "Invalid account",
            "cause": "ValidationError",
            "attempts": 1
        })
        return {
            "success": False,
            "queued": True
        }

    original_balance = account.get("balance")

    for attempt in range(1, max_attempts + 1):
        try:
            result = process_payment(event)

            return {
                **result,
                "attempts": attempt,
                "queued": False
            }

        except (ValidationError, InsufficientFundsError) as exc:
            # Business failures are not retried.
            if original_balance is not None:
                account["balance"] = original_balance

            dead_letter_queue.append({
                "event": event,
                "error": str(exc),
                "cause": type(exc).__name__,
                "attempts": attempt
            })

            return {
                "success": False,
                "account": account.get("id", "unknown"),
                "amount": event.get("amount"),
                "balance": account.get("balance"),
                "queued": True,
                "attempts": attempt,
                "error": str(exc),
                "cause": type(exc).__name__
            }

        except OSError as exc:
            # Infrastructure failure: retry up to max_attempts.
            if attempt == max_attempts:
                if original_balance is not None:
                    account["balance"] = original_balance

                dead_letter_queue.append({
                    "event": event,
                    "error": str(exc),
                    "cause": type(exc).__name__,
                    "attempts": attempt
                })

                return {
                    "success": False,
                    "account": account.get("id", "unknown"),
                    "amount": event.get("amount"),
                    "balance": account.get("balance"),
                    "queued": True,
                    "attempts": attempt,
                    "error": "Payment processing failed",
                    "cause": str(exc)
                }


# Sample reference: ACC-1076
account = {
    "id": "ACC-1076",
    "balance": 5000
}

event = {
    "account": account,
    "amount": 6000
}

result = handle_payment(event)

print(result)
print("Balance:", account["balance"])
print("Dead-letter queue:", list(dead_letter_queue))

{'success': False, 'account': 'ACC-1076', 'amount': 6000, 'balance': 5000, 'queued': True, 'attempts': 1, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError'}
Balance: 5000
Dead-letter queue: [{'event': {'account': {'id': 'ACC-1076', 'balance': 5000}, 'amount': 6000}, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError', 'attempts': 1}]


### Q77. Make batch settlement failure-isolated for banking case 1; use sample reference `ACC-1077` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


def settle_payment(event):
    account = event["account"]
    amount = event["amount"]

    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")

    # Smallest state-changing statement.
    account["balance"] -= amount

    return {
        "success": True,
        "account": account["id"],
        "amount": amount,
        "balance": account["balance"]
    }


def batch_settlement(events):
    results = []

    if not isinstance(events, list):
        return [{
            "success": False,
            "error": "Events must be a list",
            "cause": "ValidationError"
        }]

    for event in events:
        account = event.get("account") if isinstance(event, dict) else None
        original_balance = (
            account.get("balance")
            if isinstance(account, dict)
            else None
        )

        try:
            result = settle_payment(event)
            results.append(result)

        except (ValidationError, InsufficientFundsError) as exc:
            # Roll back this event only.
            if isinstance(account, dict) and original_balance is not None:
                account["balance"] = original_balance

            results.append({
                "success": False,
                "account": (
                    account.get("id", "unknown")
                    if isinstance(account, dict)
                    else "unknown"
                ),
                "amount": (
                    event.get("amount")
                    if isinstance(event, dict)
                    else None
                ),
                "balance": (
                    account.get("balance")
                    if isinstance(account, dict)
                    else None
                ),
                "error": str(exc),
                "cause": type(exc).__name__
            })

    return results


# Sample reference: ACC-1077
account_1077 = {
    "id": "ACC-1077",
    "balance": 5000
}

account_1077_b = {
    "id": "ACC-1077-B",
    "balance": 3000
}

events = [
    {
        "account": account_1077,
        "amount": 6000
    },
    {
        "account": account_1077_b,
        "amount": 1000
    }
]

results = batch_settlement(events)

for result in results:
    print(result)

print("ACC-1077 Balance:", account_1077["balance"])
print("ACC-1077-B Balance:", account_1077_b["balance"])

{'success': False, 'account': 'ACC-1077', 'amount': 6000, 'balance': 5000, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError'}
{'success': True, 'account': 'ACC-1077-B', 'amount': 1000, 'balance': 2000}
ACC-1077 Balance: 5000
ACC-1077-B Balance: 2000


### Q78. Report exact paths in nested statements for banking case 1; use sample reference `ACC-1078` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


def get_nested_value(data, path):
    current = data

    for key in path:
        try:
            current = current[key]
        except (KeyError, TypeError, IndexError) as exc:
            raise ValidationError(
                f"Invalid path: {'.'.join(path)}"
            ) from exc

    return current


def validate_account(account):
    required_paths = [
        ("id",),
        ("profile", "name"),
        ("profile", "contact", "email"),
        ("balance",)
    ]

    for path in required_paths:
        value = get_nested_value(account, path)

        if value is None or value == "":
            raise ValidationError(
                f"Empty value at path: {'.'.join(path)}"
            )

    return True


def process_account(account):
    original_balance = account.get("balance")

    try:
        validate_account(account)

        # No balance update occurs until validation succeeds.
        return {
            "success": True,
            "account": account["id"],
            "balance": account["balance"],
            "error": None,
            "path": None,
            "cause": None
        }

    except ValidationError as exc:
        # Protect the balance if a later operation had changed it.
        if original_balance is not None:
            account["balance"] = original_balance

        return {
            "success": False,
            "account": account.get("id", "unknown"),
            "balance": account.get("balance"),
            "error": str(exc),
            "path": str(exc).replace("Invalid path: ", ""),
            "cause": type(exc.__cause__).__name__
            if exc.__cause__ else type(exc).__name__
        }


# Sample reference: ACC-1078
account = {
    "id": "ACC-1078",
    "balance": 5000,
    "profile": {
        "name": "Alex",
        "contact": {
            # "email" is intentionally missing
        }
    }
}

result = process_account(account)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1078', 'balance': 5000, 'error': 'Invalid path: profile.contact.email', 'path': 'profile.contact.email', 'cause': 'KeyError'}
Balance: 5000


### Q79. Design idempotent recovery after timeout for banking case 1; use sample reference `ACC-1079` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class TimeoutError(BankingError):
    pass


processed_transactions = {}


def validate_payment(event):
    if not isinstance(event, dict):
        raise ValidationError("Invalid payment event")

    if "transaction_id" not in event:
        raise ValidationError("Missing transaction ID")

    if "account" not in event:
        raise ValidationError("Missing account")

    if "amount" not in event:
        raise ValidationError("Missing amount")

    account = event["account"]
    amount = event["amount"]

    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")


def process_payment(event, simulate_timeout=False):
    validate_payment(event)

    transaction_id = event["transaction_id"]
    account = event["account"]
    amount = event["amount"]

    # Idempotency check.
    if transaction_id in processed_transactions:
        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "status": "already_processed"
        }

    original_balance = account["balance"]

    try:
        # Smallest state-changing statement.
        account["balance"] -= amount

        # Simulate a timeout after the debit.
        if simulate_timeout:
            raise TimeoutError("Payment response timed out")

        # Record only after the transaction completes successfully.
        processed_transactions[transaction_id] = {
            "account": account["id"],
            "amount": amount
        }

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "status": "processed"
        }

    except TimeoutError as exc:
        # Roll back because the transaction was not recorded as completed.
        account["balance"] = original_balance

        raise BankingError(
            f"Payment timed out for transaction {transaction_id}"
        ) from exc


def recover_payment(event):
    try:
        return process_payment(event)

    except BankingError as exc:
        return {
            "success": False,
            "account": event["account"].get("id", "unknown"),
            "amount": event.get("amount"),
            "balance": event["account"].get("balance"),
            "error": str(exc),
            "cause": str(exc.__cause__) if exc.__cause__ else None
        }


# Sample reference: ACC-1079
account = {
    "id": "ACC-1079",
    "balance": 5000
}

event = {
    "transaction_id": "TXN-1079",
    "account": account,
    "amount": 1000
}

# First attempt: simulate a timeout.
try:
    process_payment(event, simulate_timeout=True)
except BankingError as exc:
    print({
        "success": False,
        "error": str(exc),
        "cause": str(exc.__cause__)
        if exc.__cause__ else None
    })

print("After timeout:", account["balance"])

# Recovery attempt.
result = recover_payment(event)

print(result)
print("Final balance:", account["balance"])

{'success': False, 'error': 'Payment timed out for transaction TXN-1079', 'cause': 'Payment response timed out'}
After timeout: 5000
{'success': True, 'account': 'ACC-1079', 'amount': 1000, 'balance': 4000, 'status': 'processed'}
Final balance: 4000


### Q80. Build an exception-safe reconciliation pipeline for banking case 1; use sample reference `ACC-1080` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class ReconciliationError(BankingError):
    pass


def validate_record(record):
    if not isinstance(record, dict):
        raise ValidationError("Record must be a dictionary")

    if "account" not in record or "expected_balance" not in record:
        raise ValidationError("Missing reconciliation data")

    account = record["account"]
    expected_balance = record["expected_balance"]

    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(expected_balance, (int, float)):
        raise ValidationError("Invalid expected balance")

    if isinstance(expected_balance, bool):
        raise ValidationError("Invalid expected balance")

    if expected_balance < 0:
        raise ValidationError("Expected balance cannot be negative")


def reconcile_record(record):
    validate_record(record)

    account = record["account"]
    expected_balance = record["expected_balance"]

    original_balance = account["balance"]

    try:
        # Smallest state-changing statement.
        account["balance"] = expected_balance

        # Detect a reconciliation failure.
        if account["balance"] != expected_balance:
            raise ReconciliationError("Balance reconciliation failed")

        return {
            "success": True,
            "account": account["id"],
            "old_balance": original_balance,
            "new_balance": account["balance"],
            "status": "reconciled"
        }

    except ReconciliationError as exc:
        # Roll back the partial update.
        account["balance"] = original_balance

        raise ReconciliationError(
            f"Reconciliation failed for {account['id']}"
        ) from exc


def reconciliation_pipeline(records):
    if not isinstance(records, list):
        return [{
            "success": False,
            "error": "Records must be a list",
            "cause": "ValidationError"
        }]

    results = []

    for record in records:
        account = (
            record.get("account")
            if isinstance(record, dict)
            else None
        )

        original_balance = (
            account.get("balance")
            if isinstance(account, dict)
            else None
        )

        try:
            result = reconcile_record(record)
            results.append(result)

        except ValidationError as exc:
            results.append({
                "success": False,
                "account": (
                    account.get("id", "unknown")
                    if isinstance(account, dict)
                    else "unknown"
                ),
                "balance": (
                    account.get("balance")
                    if isinstance(account, dict)
                    else None
                ),
                "error": str(exc),
                "cause": type(exc).__name__
            })

        except ReconciliationError as exc:
            # Extra protection at the pipeline level.
            if isinstance(account, dict) and original_balance is not None:
                account["balance"] = original_balance

            results.append({
                "success": False,
                "account": (
                    account.get("id", "unknown")
                    if isinstance(account, dict)
                    else "unknown"
                ),
                "balance": (
                    account.get("balance")
                    if isinstance(account, dict)
                    else None
                ),
                "error": str(exc),
                "cause": (
                    type(exc.__cause__).__name__
                    if exc.__cause__
                    else type(exc).__name__
                )
            })

    return results


# Sample reference: ACC-1080
account = {
    "id": "ACC-1080",
    "balance": 5000
}

records = [
    {
        "account": account,
        "expected_balance": 4500
    }
]

results = reconciliation_pipeline(records)

print(results)
print("Final balance:", account["balance"])

[{'success': True, 'account': 'ACC-1080', 'old_balance': 5000, 'new_balance': 4500, 'status': 'reconciled'}]
Final balance: 4500


### Q81. Design a banking exception hierarchy for banking case 2; use sample reference `ACC-1081` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    """Base class for all expected banking errors."""
    pass


class ValidationError(BankingError):
    """Invalid input or account data."""
    pass


class AccountError(BankingError):
    """Invalid or incomplete account."""
    pass


class InsufficientFundsError(BankingError):
    """Withdrawal exceeds available balance."""
    pass


class InfrastructureError(BankingError):
    """Temporary or external service failure."""
    pass


def withdraw(account, amount):
    original_balance = None

    try:
        # Account validation
        if not isinstance(account, dict):
            raise AccountError("Invalid account")

        if "id" not in account or "balance" not in account:
            raise AccountError("Missing account information")

        original_balance = account["balance"]

        # Amount validation
        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise ValidationError("Invalid amount type")

        if amount <= 0:
            raise ValidationError("Amount must be positive")

        # Business-rule validation
        if amount > account["balance"]:
            raise InsufficientFundsError("Insufficient funds")

        # State change only after validation succeeds
        account["balance"] -= amount

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None,
            "cause": None
        }

    except BankingError as exc:
        # Protect against partial balance updates
        if original_balance is not None:
            account["balance"] = original_balance

        return {
            "success": False,
            "account": account.get("id", "unknown")
            if isinstance(account, dict) else "unknown",
            "amount": amount,
            "balance": account.get("balance")
            if isinstance(account, dict) else None,
            "error": str(exc),
            "cause": type(exc).__name__
        }


# Sample reference: ACC-1081
account = {
    "id": "ACC-1081",
    "balance": 5000
}

result = withdraw(account, 6000)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1081', 'amount': 6000, 'balance': 5000, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError'}
Balance: 5000


### Q82. Make a transfer rollback-safe for banking case 2; use sample reference `ACC-1082` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class TransferError(BankingError):
    pass


def transfer(account, amount, simulate_failure=False):
    original_balance = None

    try:
        if not isinstance(account, dict):
            raise ValidationError("Invalid account")

        if "id" not in account or "balance" not in account:
            raise ValidationError("Missing account information")

        original_balance = account["balance"]

        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise ValidationError("Invalid amount type")

        if amount <= 0:
            raise ValidationError("Amount must be positive")

        if amount > account["balance"]:
            raise InsufficientFundsError("Insufficient funds")

        # Smallest risky state-changing statement
        account["balance"] -= amount

        # Simulate a failure after the balance has changed
        if simulate_failure:
            raise TransferError("Transfer confirmation failed")

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None,
            "cause": None
        }

    except BankingError as exc:
        # Roll back any partial balance update
        if original_balance is not None:
            account["balance"] = original_balance

        return {
            "success": False,
            "account": account.get("id", "unknown")
            if isinstance(account, dict) else "unknown",
            "amount": amount,
            "balance": account.get("balance")
            if isinstance(account, dict) else None,
            "error": str(exc),
            "cause": type(exc).__name__
        }


# Sample reference: ACC-1082
account = {
    "id": "ACC-1082",
    "balance": 5000
}

result = transfer(account, 1000, simulate_failure=True)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1082', 'amount': 1000, 'balance': 5000, 'error': 'Transfer confirmation failed', 'cause': 'TransferError'}
Balance: 5000


### Q83. Preserve causes across service layers for banking case 2; use sample reference `ACC-1083` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class WithdrawalServiceError(BankingError):
    pass


def validate_withdrawal(account, amount):
    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")


def account_service(account, amount):
    original_balance = account.get("balance")

    try:
        validate_withdrawal(account, amount)

        # State-changing operation
        account["balance"] -= amount

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"]
        }

    except BankingError:
        # Protect against partial updates
        if original_balance is not None:
            account["balance"] = original_balance
        raise


def withdrawal_service(account, amount):
    try:
        return account_service(account, amount)

    except BankingError as exc:
        # Add service-level context while preserving the original cause
        raise WithdrawalServiceError(
            f"Withdrawal service failed for {account.get('id', 'unknown')}"
        ) from exc


def process_withdrawal(account, amount):
    try:
        return withdrawal_service(account, amount)

    except WithdrawalServiceError as exc:
        return {
            "success": False,
            "account": account.get("id", "unknown"),
            "amount": amount,
            "balance": account.get("balance"),
            "error": str(exc),
            "cause": str(exc.__cause__),
            "cause_type": type(exc.__cause__).__name__
        }


# Sample reference: ACC-1083
account = {
    "id": "ACC-1083",
    "balance": 5000
}

result = process_withdrawal(account, 6000)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1083', 'amount': 6000, 'balance': 5000, 'error': 'Withdrawal service failed for ACC-1083', 'cause': 'Insufficient funds', 'cause_type': 'InsufficientFundsError'}
Balance: 5000


### Q84. Implement bounded exponential retry for banking case 2; use sample reference `ACC-1084` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

import time


class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class ServiceUnavailableError(BankingError):
    pass


def withdraw(account, amount, fail_times=0):
    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")

    # Smallest state-changing operation.
    account["balance"] -= amount

    if fail_times > 0:
        raise ServiceUnavailableError("Withdrawal service unavailable")

    return {
        "success": True,
        "account": account["id"],
        "amount": amount,
        "balance": account["balance"]
    }


def withdraw_with_retry(account, amount, max_retries=3, base_delay=0.1):
    original_balance = (
        account.get("balance")
        if isinstance(account, dict)
        else None
    )

    for attempt in range(max_retries + 1):
        try:
            result = withdraw(account, amount)
            return {
                **result,
                "attempts": attempt + 1
            }

        except (ValidationError, InsufficientFundsError):
            # These are deterministic business failures.
            if original_balance is not None:
                account["balance"] = original_balance
            raise

        except ServiceUnavailableError as exc:
            # Roll back the debit before retrying.
            if original_balance is not None:
                account["balance"] = original_balance

            if attempt == max_retries:
                raise BankingError(
                    f"Withdrawal failed after {max_retries + 1} attempts"
                ) from exc

            delay = base_delay * (2 ** attempt)
            time.sleep(delay)


# Sample reference: ACC-1084
account = {
    "id": "ACC-1084",
    "balance": 5000
}

try:
    result = withdraw_with_retry(account, 6000)

except BankingError as exc:
    result = {
        "success": False,
        "account": account["id"],
        "amount": 6000,
        "balance": account["balance"],
        "error": str(exc),
        "cause": (
            str(exc.__cause__)
            if exc.__cause__
            else None
        )
    }

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1084', 'amount': 6000, 'balance': 5000, 'error': 'Insufficient funds', 'cause': None}
Balance: 5000


### Q85. Create a transaction context manager for banking case 2; use sample reference `ACC-1085` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
from contextlib import contextmanager


class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class TransactionError(BankingError):
    pass


@contextmanager
def transaction(account):
    original_balance = account.get("balance")

    try:
        yield
    except BankingError:
        # Roll back any partial update.
        if original_balance is not None:
            account["balance"] = original_balance
        raise


def withdraw(account, amount, simulate_failure=False):
    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")

    try:
        with transaction(account):
            # State-changing operation
            account["balance"] -= amount

            # Simulate a failure after the debit
            if simulate_failure:
                raise TransactionError(
                    "Withdrawal confirmation failed"
                )

            return {
                "success": True,
                "account": account["id"],
                "amount": amount,
                "balance": account["balance"],
                "error": None
            }

    except BankingError as exc:
        return {
            "success": False,
            "account": account.get("id", "unknown"),
            "amount": amount,
            "balance": account.get("balance"),
            "error": str(exc),
            "cause": type(exc).__name__
        }


# Sample reference: ACC-1085
account = {
    "id": "ACC-1085",
    "balance": 5000
}

result = withdraw(account, 1000, simulate_failure=True)

print(result)
print("Balance:", account["balance"])# Write your solution here


{'success': False, 'account': 'ACC-1085', 'amount': 1000, 'balance': 5000, 'error': 'Withdrawal confirmation failed', 'cause': 'TransactionError'}
Balance: 5000


### Q86. Build a dead-letter queue for payment events for banking case 2; use sample reference `ACC-1086` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

from collections import deque


class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class PaymentServiceError(BankingError):
    pass


dead_letter_queue = deque()


def process_payment(event):
    if not isinstance(event, dict):
        raise ValidationError("Event must be a dictionary")

    if "account" not in event or "amount" not in event:
        raise ValidationError("Missing payment information")

    account = event["account"]
    amount = event["amount"]

    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")

    # Smallest risky state-changing statement.
    account["balance"] -= amount

    return {
        "success": True,
        "account": account["id"],
        "amount": amount,
        "balance": account["balance"]
    }


def handle_payment(event, max_attempts=3):
    if not isinstance(event, dict):
        dead_letter_queue.append({
            "event": event,
            "error": "Event must be a dictionary",
            "cause": "ValidationError",
            "attempts": 1
        })

        return {
            "success": False,
            "queued": True
        }

    account = event.get("account")

    if not isinstance(account, dict):
        dead_letter_queue.append({
            "event": event,
            "error": "Invalid account",
            "cause": "ValidationError",
            "attempts": 1
        })

        return {
            "success": False,
            "queued": True
        }

    original_balance = account.get("balance")

    for attempt in range(1, max_attempts + 1):
        try:
            result = process_payment(event)

            return {
                **result,
                "attempts": attempt,
                "queued": False
            }

        except (ValidationError, InsufficientFundsError) as exc:
            # Business failures are not retryable.
            if original_balance is not None:
                account["balance"] = original_balance

            dead_letter_queue.append({
                "event": event,
                "error": str(exc),
                "cause": type(exc).__name__,
                "attempts": attempt
            })

            return {
                "success": False,
                "account": account.get("id", "unknown"),
                "amount": event.get("amount"),
                "balance": account.get("balance"),
                "queued": True,
                "attempts": attempt,
                "error": str(exc),
                "cause": type(exc).__name__
            }

        except PaymentServiceError as exc:
            # Retry temporary payment-service failures.
            if original_balance is not None:
                account["balance"] = original_balance

            if attempt == max_attempts:
                dead_letter_queue.append({
                    "event": event,
                    "error": str(exc),
                    "cause": type(exc).__name__,
                    "attempts": attempt
                })

                return {
                    "success": False,
                    "account": account.get("id", "unknown"),
                    "amount": event.get("amount"),
                    "balance": account.get("balance"),
                    "queued": True,
                    "attempts": attempt,
                    "error": "Payment service failed",
                    "cause": str(exc)
                }

    return {
        "success": False,
        "queued": True
    }


# Sample reference: ACC-1086
account = {
    "id": "ACC-1086",
    "balance": 5000
}

event = {
    "account": account,
    "amount": 6000
}

result = handle_payment(event)

print(result)
print("Balance:", account["balance"])
print("Dead-letter queue:", list(dead_letter_queue))

{'success': False, 'account': 'ACC-1086', 'amount': 6000, 'balance': 5000, 'queued': True, 'attempts': 1, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError'}
Balance: 5000
Dead-letter queue: [{'event': {'account': {'id': 'ACC-1086', 'balance': 5000}, 'amount': 6000}, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError', 'attempts': 1}]


### Q87. Make batch settlement failure-isolated for banking case 2; use sample reference `ACC-1087` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


def settle_withdrawal(event):
    if not isinstance(event, dict):
        raise ValidationError("Event must be a dictionary")

    if "account" not in event or "amount" not in event:
        raise ValidationError("Missing withdrawal information")

    account = event["account"]
    amount = event["amount"]

    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")

    # Smallest state-changing statement.
    account["balance"] -= amount

    return {
        "success": True,
        "account": account["id"],
        "amount": amount,
        "balance": account["balance"]
    }


def batch_settlement(events):
    results = []

    if not isinstance(events, list):
        return [{
            "success": False,
            "error": "Events must be a list",
            "cause": "ValidationError"
        }]

    for event in events:
        account = (
            event.get("account")
            if isinstance(event, dict)
            else None
        )

        original_balance = (
            account.get("balance")
            if isinstance(account, dict)
            else None
        )

        try:
            result = settle_withdrawal(event)
            results.append(result)

        except (ValidationError, InsufficientFundsError) as exc:
            # Roll back this transaction only.
            if isinstance(account, dict) and original_balance is not None:
                account["balance"] = original_balance

            results.append({
                "success": False,
                "account": (
                    account.get("id", "unknown")
                    if isinstance(account, dict)
                    else "unknown"
                ),
                "amount": (
                    event.get("amount")
                    if isinstance(event, dict)
                    else None
                ),
                "balance": (
                    account.get("balance")
                    if isinstance(account, dict)
                    else None
                ),
                "error": str(exc),
                "cause": type(exc).__name__
            })

    return results


# Sample reference: ACC-1087
account_1087 = {
    "id": "ACC-1087",
    "balance": 5000
}

account_1087_b = {
    "id": "ACC-1087-B",
    "balance": 3000
}

events = [
    {
        "account": account_1087,
        "amount": 6000
    },
    {
        "account": account_1087_b,
        "amount": 1000
    }
]

results = batch_settlement(events)

for result in results:
    print(result)

print("ACC-1087 Balance:", account_1087["balance"])
print("ACC-1087-B Balance:", account_1087_b["balance"])

{'success': False, 'account': 'ACC-1087', 'amount': 6000, 'balance': 5000, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError'}
{'success': True, 'account': 'ACC-1087-B', 'amount': 1000, 'balance': 2000}
ACC-1087 Balance: 5000
ACC-1087-B Balance: 2000


### Q88. Report exact paths in nested statements for banking case 2; use sample reference `ACC-1088` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


def get_nested_value(data, path):
    current = data

    for key in path:
        try:
            current = current[key]
        except (KeyError, TypeError, IndexError) as exc:
            raise ValidationError(
                f"Invalid path: {'.'.join(path)}"
            ) from exc

    return current


def validate_withdrawal_record(record):
    required_paths = [
        ("account", "id"),
        ("account", "customer", "name"),
        ("account", "customer", "contact", "phone"),
        ("account", "balance"),
        ("withdrawal", "amount")
    ]

    for path in required_paths:
        value = get_nested_value(record, path)

        if value is None or value == "":
            raise ValidationError(
                f"Empty value at path: {'.'.join(path)}"
            )

    amount = get_nested_value(
        record,
        ("withdrawal", "amount")
    )

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError(
            "Invalid amount at path: withdrawal.amount"
        )

    if amount <= 0:
        raise ValidationError(
            "Amount must be positive at path: withdrawal.amount"
        )


def process_withdrawal(record):
    account = record.get("account", {})
    original_balance = (
        account.get("balance")
        if isinstance(account, dict)
        else None
    )

    try:
        validate_withdrawal_record(record)

        account["balance"] -= record["withdrawal"]["amount"]

        return {
            "success": True,
            "account": account["id"],
            "amount": record["withdrawal"]["amount"],
            "balance": account["balance"],
            "error": None,
            "path": None,
            "cause": None
        }

    except ValidationError as exc:
        # Protect the balance if a state change had occurred.
        if original_balance is not None:
            account["balance"] = original_balance

        return {
            "success": False,
            "account": (
                account.get("id", "unknown")
                if isinstance(account, dict)
                else "unknown"
            ),
            "amount": (
                record.get("withdrawal", {}).get("amount")
                if isinstance(record, dict)
                else None
            ),
            "balance": (
                account.get("balance")
                if isinstance(account, dict)
                else None
            ),
            "error": str(exc),
            "path": extract_path(str(exc)),
            "cause": (
                type(exc.__cause__).__name__
                if exc.__cause__
                else type(exc).__name__
            )
        }


def extract_path(message):
    if "Invalid path: " in message:
        return message.split("Invalid path: ", 1)[1]

    if " at path: " in message:
        return message.split(" at path: ", 1)[1]

    return None


# Sample reference: ACC-1088
record = {
    "account": {
        "id": "ACC-1088",
        "balance": 5000,
        "customer": {
            "name": "Alex",
            "contact": {
                # "phone" is intentionally missing
            }
        }
    },
    "withdrawal": {
        "amount": 1000
    }
}

result = process_withdrawal(record)

print(result)
print("Balance:", record["account"]["balance"])

{'success': False, 'account': 'ACC-1088', 'amount': 1000, 'balance': 5000, 'error': 'Invalid path: account.customer.contact.phone', 'path': 'account.customer.contact.phone', 'cause': 'KeyError'}
Balance: 5000


### Q89. Design idempotent recovery after timeout for banking case 2; use sample reference `ACC-1089` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class PaymentTimeoutError(BankingError):
    pass


processed_transactions = {}


def validate_withdrawal(event):
    if not isinstance(event, dict):
        raise ValidationError("Event must be a dictionary")

    if "transaction_id" not in event:
        raise ValidationError("Missing transaction ID")

    if "account" not in event:
        raise ValidationError("Missing account")

    if "amount" not in event:
        raise ValidationError("Missing amount")

    account = event["account"]
    amount = event["amount"]

    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")


def withdraw(event, simulate_timeout=False):
    validate_withdrawal(event)

    transaction_id = event["transaction_id"]
    account = event["account"]
    amount = event["amount"]

    if transaction_id in processed_transactions:
        previous = processed_transactions[transaction_id]

        return {
            "success": True,
            "account": previous["account"],
            "amount": previous["amount"],
            "balance": account["balance"],
            "status": "already_processed"
        }

    original_balance = account["balance"]

    try:
        account["balance"] -= amount

        if simulate_timeout:
            raise PaymentTimeoutError(
                "Withdrawal confirmation timed out"
            )

        processed_transactions[transaction_id] = {
            "account": account["id"],
            "amount": amount
        }

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "status": "processed"
        }

    except PaymentTimeoutError as exc:
        account["balance"] = original_balance

        raise BankingError(
            f"Withdrawal timed out for {transaction_id}"
        ) from exc


def recover_withdrawal(event):
    try:
        return withdraw(event)

    except BankingError as exc:
        return {
            "success": False,
            "account": event["account"].get("id", "unknown"),
            "amount": event.get("amount"),
            "balance": event["account"].get("balance"),
            "error": str(exc),
            "cause": (
                str(exc.__cause__)
                if exc.__cause__
                else None
            )
        }


# Sample reference: ACC-1089
account = {
    "id": "ACC-1089",
    "balance": 5000
}

event = {
    "transaction_id": "TXN-1089",
    "account": account,
    "amount": 1000
}

try:
    withdraw(event, simulate_timeout=True)
except BankingError as exc:
    print({
        "success": False,
        "account": account["id"],
        "amount": event["amount"],
        "balance": account["balance"],
        "error": str(exc),
        "cause": str(exc.__cause__)
    })

print("After timeout:", account["balance"])

result = recover_withdrawal(event)

print(result)
print("Final balance:", account["balance"])

duplicate = recover_withdrawal(event)

print(duplicate)
print("Balance after duplicate:", account["balance"])

{'success': False, 'account': 'ACC-1089', 'amount': 1000, 'balance': 5000, 'error': 'Withdrawal timed out for TXN-1089', 'cause': 'Withdrawal confirmation timed out'}
After timeout: 5000
{'success': True, 'account': 'ACC-1089', 'amount': 1000, 'balance': 4000, 'status': 'processed'}
Final balance: 4000
{'success': True, 'account': 'ACC-1089', 'amount': 1000, 'balance': 4000, 'status': 'already_processed'}
Balance after duplicate: 4000


### Q90. Build an exception-safe reconciliation pipeline for banking case 2; use sample reference `ACC-1090` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class ReconciliationError(BankingError):
    pass


def validate_record(record):
    if not isinstance(record, dict):
        raise ValidationError("Record must be a dictionary")

    if "account" not in record:
        raise ValidationError("Missing account")

    if "expected_balance" not in record:
        raise ValidationError("Missing expected balance")

    account = record["account"]
    expected_balance = record["expected_balance"]

    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(expected_balance, (int, float)):
        raise ValidationError("Invalid expected balance")

    if isinstance(expected_balance, bool):
        raise ValidationError("Invalid expected balance")

    if expected_balance < 0:
        raise ValidationError("Expected balance cannot be negative")


def reconcile_record(record, simulate_failure=False):
    validate_record(record)

    account = record["account"]
    expected_balance = record["expected_balance"]

    original_balance = account["balance"]

    try:
        # Smallest state-changing statement
        account["balance"] = expected_balance

        if simulate_failure:
            raise ReconciliationError(
                "Reconciliation confirmation failed"
            )

        if account["balance"] != expected_balance:
            raise ReconciliationError(
                "Balance reconciliation failed"
            )

        return {
            "success": True,
            "account": account["id"],
            "old_balance": original_balance,
            "new_balance": account["balance"],
            "error": None,
            "cause": None
        }

    except ReconciliationError as exc:
        # Roll back partial update
        account["balance"] = original_balance

        raise ReconciliationError(
            f"Reconciliation failed for {account['id']}"
        ) from exc


def reconciliation_pipeline(records):
    if not isinstance(records, list):
        return [{
            "success": False,
            "error": "Records must be a list",
            "cause": "ValidationError"
        }]

    results = []

    for record in records:
        account = (
            record.get("account")
            if isinstance(record, dict)
            else None
        )

        original_balance = (
            account.get("balance")
            if isinstance(account, dict)
            else None
        )

        try:
            result = reconcile_record(record)
            results.append(result)

        except ValidationError as exc:
            results.append({
                "success": False,
                "account": (
                    account.get("id", "unknown")
                    if isinstance(account, dict)
                    else "unknown"
                ),
                "balance": (
                    account.get("balance")
                    if isinstance(account, dict)
                    else None
                ),
                "error": str(exc),
                "cause": type(exc).__name__
            })

        except ReconciliationError as exc:
            # Extra protection against partial updates
            if isinstance(account, dict) and original_balance is not None:
                account["balance"] = original_balance

            results.append({
                "success": False,
                "account": (
                    account.get("id", "unknown")
                    if isinstance(account, dict)
                    else "unknown"
                ),
                "balance": (
                    account.get("balance")
                    if isinstance(account, dict)
                    else None
                ),
                "error": str(exc),
                "cause": (
                    type(exc.__cause__).__name__
                    if exc.__cause__
                    else type(exc).__name__
                )
            })


    return results


# Sample reference: ACC-1090
account = {
    "id": "ACC-1090",
    "balance": 5000
}

records = [
    {
        "account": account,
        "expected_balance": 4500
    }
]

results = reconciliation_pipeline(records)

for result in results:
    print(result)

print("Balance:", account["balance"])

{'success': True, 'account': 'ACC-1090', 'old_balance': 5000, 'new_balance': 4500, 'error': None, 'cause': None}
Balance: 4500


### Q91. Design a banking exception hierarchy for banking case 3; use sample reference `ACC-1091` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    """Base exception for expected banking errors."""
    pass


class ValidationError(BankingError):
    """Invalid input or data."""
    pass


class AccountError(BankingError):
    """Invalid or incomplete account."""
    pass


class InsufficientFundsError(BankingError):
    """Requested amount exceeds available balance."""
    pass


class InfrastructureError(BankingError):
    """External service or infrastructure failure."""
    pass


def transfer(account, amount):
    original_balance = None

    try:
        if not isinstance(account, dict):
            raise AccountError("Invalid account")

        if "id" not in account or "balance" not in account:
            raise AccountError("Missing account information")

        original_balance = account["balance"]

        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise ValidationError("Invalid amount type")

        if amount <= 0:
            raise ValidationError("Amount must be positive")

        if amount > account["balance"]:
            raise InsufficientFundsError("Insufficient funds")

        # Smallest state-changing statement
        account["balance"] -= amount

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None,
            "cause": None
        }

    except BankingError as exc:
        # Protect the balance from partial updates.
        if original_balance is not None:
            account["balance"] = original_balance

        return {
            "success": False,
            "account": (
                account.get("id", "unknown")
                if isinstance(account, dict)
                else "unknown"
            ),
            "amount": amount,
            "balance": (
                account.get("balance")
                if isinstance(account, dict)
                else None
            ),
            "error": str(exc),
            "cause": type(exc).__name__
        }


# Sample reference: ACC-1091
account = {
    "id": "ACC-1091",
    "balance": 5000
}

result = transfer(account, 6000)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1091', 'amount': 6000, 'balance': 5000, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError'}
Balance: 5000


### Q92. Make a transfer rollback-safe for banking case 3; use sample reference `ACC-1092` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class TransferError(BankingError):
    pass


def transfer(account, amount, simulate_failure=False):
    original_balance = None

    try:
        if not isinstance(account, dict):
            raise ValidationError("Invalid account")

        if "id" not in account or "balance" not in account:
            raise ValidationError("Missing account information")

        original_balance = account["balance"]

        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise ValidationError("Invalid amount type")

        if amount <= 0:
            raise ValidationError("Amount must be positive")

        if amount > account["balance"]:
            raise InsufficientFundsError("Insufficient funds")

        # Smallest state-changing statement
        account["balance"] -= amount

        # Simulate a failure after the balance changed
        if simulate_failure:
            raise TransferError("Transfer confirmation failed")

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None,
            "cause": None
        }

    except BankingError as exc:
        # Roll back partial balance update
        if original_balance is not None:
            account["balance"] = original_balance

        return {
            "success": False,
            "account": (
                account.get("id", "unknown")
                if isinstance(account, dict)
                else "unknown"
            ),
            "amount": amount,
            "balance": (
                account.get("balance")
                if isinstance(account, dict)
                else None
            ),
            "error": str(exc),
            "cause": type(exc).__name__
        }


# Sample reference: ACC-1092
account = {
    "id": "ACC-1092",
    "balance": 5000
}

result = transfer(
    account,
    1000,
    simulate_failure=True
)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1092', 'amount': 1000, 'balance': 5000, 'error': 'Transfer confirmation failed', 'cause': 'TransferError'}
Balance: 5000


### Q93. Preserve causes across service layers for banking case 3; use sample reference `ACC-1093` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class TransferServiceError(BankingError):
    pass


def validate_transfer(account, amount):
    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")


def account_service(account, amount):
    original_balance = account.get("balance")

    try:
        validate_transfer(account, amount)

        # Smallest state-changing statement
        account["balance"] -= amount

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"]
        }

    except BankingError:
        # Protect against partial balance updates
        if original_balance is not None:
            account["balance"] = original_balance
        raise


def transfer_service(account, amount):
    try:
        return account_service(account, amount)

    except BankingError as exc:
        # Add context while preserving the original cause
        raise TransferServiceError(
            f"Transfer service failed for {account.get('id', 'unknown')}"
        ) from exc


def process_transfer(account, amount):
    try:
        return transfer_service(account, amount)

    except TransferServiceError as exc:
        return {
            "success": False,
            "account": account.get("id", "unknown"),
            "amount": amount,
            "balance": account.get("balance"),
            "error": str(exc),
            "cause": str(exc.__cause__),
            "cause_type": type(exc.__cause__).__name__
        }


# Sample reference: ACC-1093
account = {
    "id": "ACC-1093",
    "balance": 5000
}

result = process_transfer(account, 6000)

print(result)
print("Balance:", account["balance"])

{'success': False, 'account': 'ACC-1093', 'amount': 6000, 'balance': 5000, 'error': 'Transfer service failed for ACC-1093', 'cause': 'Insufficient funds', 'cause_type': 'InsufficientFundsError'}
Balance: 5000


### Q94. Implement bounded exponential retry for banking case 3; use sample reference `ACC-1094` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
import time


class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class InfrastructureError(BankingError):
    pass


def transfer(account, amount, fail_times=0):
    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")

    # Smallest state-changing statement
    account["balance"] -= amount

    if fail_times > 0:
        raise InfrastructureError("Transfer service temporarily unavailable")

    return {
        "success": True,
        "account": account["id"],
        "amount": amount,
        "balance": account["balance"]
    }


def transfer_with_retry(
    account,
    amount,
    max_retries=3,
    base_delay=0.1,
    fail_times=0
):
    original_balance = (
        account.get("balance")
        if isinstance(account, dict)
        else None
    )

    remaining_failures = fail_times

    for attempt in range(max_retries + 1):
        try:
            result = transfer(
                account,
                amount,
                fail_times=remaining_failures
            )

            return {
                **result,
                "attempts": attempt + 1
            }

        except (ValidationError, InsufficientFundsError):
            # Permanent business/input failures are not retried.
            if original_balance is not None:
                account["balance"] = original_balance
            raise

        except InfrastructureError as exc:
            # Roll back before retrying.
            if original_balance is not None:
                account["balance"] = original_balance

            if remaining_failures > 0:
                remaining_failures -= 1

            if attempt == max_retries:
                raise BankingError(
                    f"Transfer failed after {attempt + 1} attempts"
                ) from exc

            delay = base_delay * (2 ** attempt)
            print(
                f"Attempt {attempt + 1} failed. "
                f"Retrying in {delay:.1f} seconds..."
            )

            # In a real application, use time.sleep(delay).
            # Kept disabled here to make testing fast.


# Sample reference: ACC-1094
account = {
    "id": "ACC-1094",
    "balance": 5000
}

try:
    result = transfer_with_retry(
        account,
        1000,
        max_retries=3,
        base_delay=0.1,
        fail_times=2
    )

    print(result)

except BankingError as exc:
    print({
        "success": False,
        "account": account["id"],
        "amount": 1000,
        "balance": account["balance"],
        "error": str(exc),
        "cause": (
            str(exc.__cause__)
            if exc.__cause__
            else None
        )
    })

print("Final balance:", account["balance"])


Attempt 1 failed. Retrying in 0.1 seconds...
Attempt 2 failed. Retrying in 0.2 seconds...
{'success': True, 'account': 'ACC-1094', 'amount': 1000, 'balance': 4000, 'attempts': 3}
Final balance: 4000


### Q95. Create a transaction context manager for banking case 3; use sample reference `ACC-1095` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class TransactionError(BankingError):
    pass


class Transaction:
    def __init__(self, account):
        self.account = account
        self.original_balance = None

    def __enter__(self):
        if not isinstance(self.account, dict):
            raise ValidationError("Invalid account")

        if "id" not in self.account or "balance" not in self.account:
            raise ValidationError("Missing account information")

        self.original_balance = self.account["balance"]
        return self

    def withdraw(self, amount):
        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise ValidationError("Invalid amount type")

        if amount <= 0:
            raise ValidationError("Amount must be positive")

        if amount > self.account["balance"]:
            raise InsufficientFundsError("Insufficient funds")

        # Smallest state-changing statement
        self.account["balance"] -= amount

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is not None:
            # Roll back any partial update.
            self.account["balance"] = self.original_balance

            if issubclass(exc_type, BankingError):
                return False

        return False


def perform_transfer(account, amount, simulate_failure=False):
    try:
        with Transaction(account) as transaction:
            transaction.withdraw(amount)

            if simulate_failure:
                raise TransactionError(
                    "Transaction confirmation failed"
                )

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None,
            "cause": None
        }

    except BankingError as exc:
        return {
            "success": False,
            "account": (
                account.get("id", "unknown")
                if isinstance(account, dict)
                else "unknown"
            ),
            "amount": amount,
            "balance": (
                account.get("balance")
                if isinstance(account, dict)
                else None
            ),
            "error": str(exc),
            "cause": (
                type(exc.__cause__).__name__
                if exc.__cause__
                else type(exc).__name__
            )
        }


# Sample reference: ACC-1095
account = {
    "id": "ACC-1095",
    "balance": 5000
}

result = perform_transfer(
    account,
    1000,
    simulate_failure=True
)

print(result)
print("Final balance:", account["balance"])

{'success': False, 'account': 'ACC-1095', 'amount': 1000, 'balance': 5000, 'error': 'Transaction confirmation failed', 'cause': 'TransactionError'}
Final balance: 5000


### Q96. Build a dead-letter queue for payment events for banking case 3; use sample reference `ACC-1096` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

from collections import deque


class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class InfrastructureError(BankingError):
    pass


dead_letter_queue = deque()


def validate_event(event):
    if not isinstance(event, dict):
        raise ValidationError("Event must be a dictionary")

    if "transaction_id" not in event:
        raise ValidationError("Missing transaction ID")

    if "account" not in event:
        raise ValidationError("Missing account")

    if "amount" not in event:
        raise ValidationError("Missing amount")

    account = event["account"]
    amount = event["amount"]

    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")


def process_payment(event, simulate_failure=False):
    validate_event(event)

    account = event["account"]
    amount = event["amount"]

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")

    original_balance = account["balance"]

    try:
        # Smallest state-changing statement
        account["balance"] -= amount

        if simulate_failure:
            raise InfrastructureError(
                "Payment service temporarily unavailable"
            )

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"]
        }

    except InfrastructureError:
        # Protect against a partial debit.
        account["balance"] = original_balance
        raise


def handle_payment(event, max_retries=2, simulate_failure=False):
    if not isinstance(event, dict):
        dead_letter_queue.append({
            "event": event,
            "error": "Event must be a dictionary",
            "cause": "ValidationError",
            "attempts": 1
        })

        return {
            "success": False,
            "queued": True
        }

    account = event.get("account")

    original_balance = (
        account.get("balance")
        if isinstance(account, dict)
        else None
    )

    for attempt in range(1, max_retries + 2):
        try:
            result = process_payment(
                event,
                simulate_failure=simulate_failure
            )

            result["attempts"] = attempt
            result["queued"] = False

            return result

        except ValidationError as exc:
            dead_letter_queue.append({
                "event": event,
                "error": str(exc),
                "cause": type(exc).__name__,
                "attempts": attempt
            })

            return {
                "success": False,
                "queued": True,
                "attempts": attempt,
                "error": str(exc),
                "cause": type(exc).__name__
            }

        except InsufficientFundsError as exc:
            dead_letter_queue.append({
                "event": event,
                "error": str(exc),
                "cause": type(exc).__name__,
                "attempts": attempt
            })

            if original_balance is not None:
                account["balance"] = original_balance

            return {
                "success": False,
                "account": account.get("id", "unknown"),
                "amount": event.get("amount"),
                "balance": account.get("balance"),
                "queued": True,
                "attempts": attempt,
                "error": str(exc),
                "cause": type(exc).__name__
            }

        except InfrastructureError as exc:
            if original_balance is not None:
                account["balance"] = original_balance

            if attempt == max_retries + 1:
                dead_letter_queue.append({
                    "event": event,
                    "error": str(exc),
                    "cause": type(exc).__name__,
                    "attempts": attempt
                })

                return {
                    "success": False,
                    "account": account.get("id", "unknown"),
                    "amount": event.get("amount"),
                    "balance": account.get("balance"),
                    "queued": True,
                    "attempts": attempt,
                    "error": "Payment service failed",
                    "cause": str(exc)
                }

            print(
                f"Attempt {attempt} failed. "
                f"Retrying..."
            )


# Sample reference: ACC-1096
account = {
    "id": "ACC-1096",
    "balance": 5000
}

event = {
    "transaction_id": "TXN-1096",
    "account": account,
    "amount": 6000
}

result = handle_payment(event)

print(result)
print("Balance:", account["balance"])
print("Dead-letter queue:", list(dead_letter_queue))

{'success': False, 'account': 'ACC-1096', 'amount': 6000, 'balance': 5000, 'queued': True, 'attempts': 1, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError'}
Balance: 5000
Dead-letter queue: [{'event': {'transaction_id': 'TXN-1096', 'account': {'id': 'ACC-1096', 'balance': 5000}, 'amount': 6000}, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError', 'attempts': 1}]


### Q97. Make batch settlement failure-isolated for banking case 3; use sample reference `ACC-1097` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


class InsufficientFundsError(BankingError):
    pass


class SettlementError(BankingError):
    pass


def settle_payment(account, amount, simulate_failure=False):
    if not isinstance(account, dict):
        raise ValidationError("Invalid account")

    if "id" not in account or "balance" not in account:
        raise ValidationError("Missing account information")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError("Invalid amount type")

    if amount <= 0:
        raise ValidationError("Amount must be positive")

    if amount > account["balance"]:
        raise InsufficientFundsError("Insufficient funds")

    original_balance = account["balance"]

    try:
        # Smallest state-changing statement
        account["balance"] -= amount

        if simulate_failure:
            raise SettlementError("Settlement failed")

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None,
            "cause": None
        }

    except SettlementError as exc:
        account["balance"] = original_balance
        raise SettlementError(
            f"Settlement failed for {account['id']}"
        ) from exc


def batch_settlement(events):
    if not isinstance(events, list):
        return [{
            "success": False,
            "account": "unknown",
            "error": "Events must be a list",
            "cause": "ValidationError"
        }]

    results = []

    for event in events:
        if not isinstance(event, dict):
            results.append({
                "success": False,
                "account": "unknown",
                "amount": None,
                "balance": None,
                "error": "Event must be a dictionary",
                "cause": "ValidationError"
            })
            continue

        account = event.get("account")
        amount = event.get("amount")
        simulate_failure = event.get("simulate_failure", False)

        original_balance = (
            account.get("balance")
            if isinstance(account, dict)
            else None
        )

        try:
            result = settle_payment(
                account,
                amount,
                simulate_failure
            )
            results.append(result)

        except (ValidationError, InsufficientFundsError) as exc:
            results.append({
                "success": False,
                "account": (
                    account.get("id", "unknown")
                    if isinstance(account, dict)
                    else "unknown"
                ),
                "amount": amount,
                "balance": (
                    account.get("balance")
                    if isinstance(account, dict)
                    else None
                ),
                "error": str(exc),
                "cause": type(exc).__name__
            })

        except SettlementError as exc:
            if (
                isinstance(account, dict)
                and original_balance is not None
            ):
                account["balance"] = original_balance

            results.append({
                "success": False,
                "account": account.get("id", "unknown"),
                "amount": amount,
                "balance": account.get("balance"),
                "error": str(exc),
                "cause": (
                    type(exc.__cause__).__name__
                    if exc.__cause__
                    else type(exc).__name__
                )
            })

    return results


# Sample reference: ACC-1097
account_1097 = {
    "id": "ACC-1097",
    "balance": 5000
}

account_1097_b = {
    "id": "ACC-1097-B",
    "balance": 3000
}

events = [
    {
        "account": account_1097,
        "amount": 1000,
        "simulate_failure": True
    },
    {
        "account": account_1097_b,
        "amount": 500
    },
    {
        "account": account_1097,
        "amount": 6000
    }
]

results = batch_settlement(events)

for result in results:
    print(result)

print("ACC-1097 Balance:", account_1097["balance"])
print("ACC-1097-B Balance:", account_1097_b["balance"])

{'success': False, 'account': 'ACC-1097', 'amount': 1000, 'balance': 5000, 'error': 'Settlement failed for ACC-1097', 'cause': 'SettlementError'}
{'success': True, 'account': 'ACC-1097-B', 'amount': 500, 'balance': 2500, 'error': None, 'cause': None}
{'success': False, 'account': 'ACC-1097', 'amount': 6000, 'balance': 5000, 'error': 'Insufficient funds', 'cause': 'InsufficientFundsError'}
ACC-1097 Balance: 5000
ACC-1097-B Balance: 2500


### Q98. Report exact paths in nested statements for banking case 3; use sample reference `ACC-1098` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:

class BankingError(Exception):
    pass


class ValidationError(BankingError):
    pass


def get_path(data, path):
    current = data
    current_path = []

    for key in path:
        current_path.append(str(key))
        path_text = ".".join(current_path)

        try:
            if isinstance(current, dict):
                current = current[key]
            elif isinstance(current, list):
                current = current[key]
            else:
                raise TypeError(
                    f"Cannot access '{key}' at {path_text}"
                )

        except (KeyError, IndexError, TypeError) as exc:
            raise ValidationError(
                f"Invalid path: {path_text}"
            ) from exc

    return current


def validate_payment(record):
    if not isinstance(record, dict):
        raise ValidationError("Payment record must be a dictionary")

    required_paths = [
        ("account", "id"),
        ("account", "customer", "name"),
        ("account", "customer", "contact", "phone"),
        ("account", "balance"),
        ("payment", "amount")
    ]

    for path in required_paths:
        value = get_path(record, path)

        if value is None or value == "":
            raise ValidationError(
                f"Empty value at path: {'.'.join(path)}"
            )

    amount = get_path(record, ("payment", "amount"))

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise ValidationError(
            "Invalid amount at path: payment.amount"
        )

    if amount <= 0:
        raise ValidationError(
            "Amount must be positive at path: payment.amount"
        )


def process_payment(record):
    account = (
        record.get("account")
        if isinstance(record, dict)
        else None
    )

    original_balance = (
        account.get("balance")
        if isinstance(account, dict)
        else None
    )

    try:
        validate_payment(record)

        account = record["account"]
        amount = record["payment"]["amount"]

        if amount > account["balance"]:
            raise ValidationError(
                "Insufficient funds at path: account.balance"
            )

        # Smallest state-changing statement
        account["balance"] -= amount

        return {
            "success": True,
            "account": account["id"],
            "amount": amount,
            "balance": account["balance"],
            "error": None,
            "path": None,
            "cause": None
        }

    except ValidationError as exc:
        if (
            isinstance(account, dict)
            and original_balance is not None
        ):
            account["balance"] = original_balance

        message = str(exc)

        if "Invalid path: " in message:
            path = message.split("Invalid path: ", 1)[1]
        elif " at path: " in message:
            path = message.split(" at path: ", 1)[1]
        else:
            path = None

        return {
            "success": False,
            "account": (
                account.get("id", "unknown")
                if isinstance(account, dict)
                else "unknown"
            ),
            "amount": (
                record.get("payment", {}).get("amount")
                if isinstance(record, dict)
                and isinstance(record.get("payment"), dict)
                else None
            ),
            "balance": (
                account.get("balance")
                if isinstance(account, dict)
                else None
            ),
            "error": message,
            "path": path,
            "cause": (
                type(exc.__cause__).__name__
                if exc.__cause__
                else type(exc).__name__
            )
        }


# Sample reference: ACC-1098
record = {
    "account": {
        "id": "ACC-1098",
        "balance": 5000,
        "customer": {
            "name": "Alex",
            "contact": {
                # phone is intentionally missing
            }
        }
    },
    "payment": {
        "amount": 1000
    }
}

result = process_payment(record)

print(result)
print("Balance:", record["account"]["balance"])

{'success': False, 'account': 'ACC-1098', 'amount': 1000, 'balance': 5000, 'error': 'Invalid path: account.customer.contact.phone', 'path': 'account.customer.contact.phone', 'cause': 'KeyError'}
Balance: 5000


### Q99. Design idempotent recovery after timeout for banking case 3; use sample reference `ACC-1099` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [38]:
class PaymentTimeoutError(Exception):
    pass


def recover_payment(account, amount, transaction_id, gateway):
    if not isinstance(account, dict) or "id" not in account or "balance" not in account:
        raise ValueError("Invalid account")

    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise TypeError("Amount must be numeric")

    if amount <= 0:
        raise ValueError("Amount must be positive")

    if amount > account["balance"]:
        raise ValueError("Insufficient funds")

    # Idempotency check: do not process the same transaction twice.
    if transaction_id in account.get("completed_transactions", set()):
        return {
            "success": True,
            "status": "already_processed",
            "transaction_id": transaction_id,
            "balance": account["balance"]
        }

    try:
        # Smallest risky operation: contacting the gateway.
        gateway.charge(account["id"], amount)

    except TimeoutError as exc:
        # Preserve the original timeout cause.
        raise PaymentTimeoutError(
            f"Payment timed out for {account['id']}"
        ) from exc

    # Update balance only after confirmed successful payment.
    account["balance"] -= amount
    account.setdefault("completed_transactions", set()).add(transaction_id)

    return {
        "success": True,
        "status": "processed",
        "transaction_id": transaction_id,
        "balance": account["balance"]
    }


# Sample reference: ACC-1099
class Gateway:
    def charge(self, account_id, amount):
        return True


account = {
    "id": "ACC-1099",
    "balance": 5000,
    "completed_transactions": set()
}

result = recover_payment(account, 1000, "TXN-1099", Gateway())
print(result)

# Safe recovery/retry using the same transaction ID
result = recover_payment(account, 1000, "TXN-1099", Gateway())
print(result)

{'success': True, 'status': 'processed', 'transaction_id': 'TXN-1099', 'balance': 4000}
{'success': True, 'status': 'already_processed', 'transaction_id': 'TXN-1099', 'balance': 4000}


### Q100. Build an exception-safe reconciliation pipeline for banking case 3; use sample reference `ACC-1100` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [39]:

class ReconciliationError(Exception):
    pass


def reconcile(account, transactions):
    if not isinstance(account, dict):
        raise TypeError("account must be a dictionary")

    if not isinstance(transactions, list):
        raise TypeError("transactions must be a list")

    if "id" not in account or "balance" not in account:
        raise ValueError("Invalid account")

    successful = []
    failed = []

    for transaction in transactions:
        try:
            if not isinstance(transaction, dict):
                raise ValueError("Invalid transaction record")

            amount = transaction["amount"]

            if not isinstance(amount, (int, float)) or isinstance(amount, bool):
                raise ValueError("Invalid transaction amount")

            if amount < 0:
                raise ValueError("Amount cannot be negative")

            # Smallest risky operation: applying the transaction.
            new_balance = account["balance"] + amount

            # Commit only after successful validation/calculation.
            account["balance"] = new_balance

            successful.append({
                "transaction": transaction,
                "status": "reconciled"
            })

        except (ValueError, KeyError) as exc:
            error = ReconciliationError(
                f"Failed to reconcile transaction for {account['id']}"
            )

            failed.append({
                "transaction": transaction,
                "error": str(error),
                "cause": str(exc)
            })

    return {
        "account": account["id"],
        "balance": account["balance"],
        "successful": successful,
        "failed": failed
    }


# Sample reference: ACC-1100
account = {
    "id": "ACC-1100",
    "balance": 5000
}

transactions = [
    {"id": "T1", "amount": 1000},
    {"id": "T2", "amount": 500},
    {"id": "T3", "amount": "invalid"},
    {"id": "T4", "amount": -200}
]

result = reconcile(account, transactions)

print(result)

{'account': 'ACC-1100', 'balance': 6500, 'successful': [{'transaction': {'id': 'T1', 'amount': 1000}, 'status': 'reconciled'}, {'transaction': {'id': 'T2', 'amount': 500}, 'status': 'reconciled'}], 'failed': [{'transaction': {'id': 'T3', 'amount': 'invalid'}, 'error': 'Failed to reconcile transaction for ACC-1100', 'cause': 'Invalid transaction amount'}, {'transaction': {'id': 'T4', 'amount': -200}, 'error': 'Failed to reconcile transaction for ACC-1100', 'cause': 'Amount cannot be negative'}]}
